COPT solver for dummy file

In [3]:
"""
NZTA envelope-first optimiser – COPT (Fixed Funding + Piecewise Soft Cap) v94.21 (DUMMY DATASET ADAPTATION)

Changes made for dummy_cost_benefits.xlsx (while preserving solve/model logic):
──────────────────────────────────────────────────────────────────────────────
1) DATA SOURCE:
   - Reads from: dummy_cost_benefits.xlsx
   - Costs are in sheet: "Costs"
   - Benefits are in sheet: "Benefits"

2) BENEFITS FORMAT:
   - Accepts t+0 ... t+40 (or any t+N set, with gaps handled robustly)
   - Accepts extra descriptor columns (Activity Class, Region, Tier, etc.) and ignores them.

3) COSTS FORMAT:
   - Robust parsing for thousands separators (e.g. "106,839,620.21") and dash placeholders (" - ")
   - Uses provided "Duration" where available (fallback: non-zero span).

4) HORIZON:
   - Programme / PV horizon set to 40 years (START_FY..START_FY+39)
   - Benefits PV is computed over the same 40-year window (as before, via disc_vec length = Tfine)

All other optimiser logic is unchanged.
"""

from __future__ import annotations

import os
import sys
import time
import re
import math
import hashlib
import random
import pickle
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Iterable, Any
from collections import defaultdict

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────────
# THREADING CONTROL
# ─────────────────────────────────────────────────────────────────────
# If you set FORCE_SINGLE_THREAD=True, we force BLAS + COPT to 1 thread.
FORCE_SINGLE_THREAD = False

if FORCE_SINGLE_THREAD:
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"
    os.environ["COPT_NUM_THREADS"] = "1"


def _cpu_threads_default() -> int:
    n = os.cpu_count() or 1
    try:
        n = int(n)
    except Exception:
        n = 1
    return max(1, n)


# Environment overrides (recommended):
#   set NZTA_COPT_THREADS=20
#   set NZTA_COPT_TASKS=20
_SOLVER_THREADS_ENV = os.environ.get("NZTA_COPT_THREADS", "").strip()
_SOLVER_TASKS_ENV = os.environ.get("NZTA_COPT_TASKS", "").strip()

try:
    SOLVER_THREADS_DEFAULT = int(_SOLVER_THREADS_ENV) if _SOLVER_THREADS_ENV else _cpu_threads_default()
except Exception:
    SOLVER_THREADS_DEFAULT = _cpu_threads_default()

try:
    SOLVER_TASKS_DEFAULT = int(_SOLVER_TASKS_ENV) if _SOLVER_TASKS_ENV else SOLVER_THREADS_DEFAULT
except Exception:
    SOLVER_TASKS_DEFAULT = SOLVER_THREADS_DEFAULT

# --- COPT --------------------------------------------------------------------
try:
    import coptpy as co
except Exception as e:
    raise RuntimeError(
        "coptpy is required. Please ensure COPT is installed and licensed "
        "(pip install coptpy) and COPT_HOME is configured."
    ) from e

COPT = co.COPT

# ─────────────────────────────────────────────────────────────────────
# CALLBACK CONSTANT DISCOVERY (kept for compatibility, but unused)
# ─────────────────────────────────────────────────────────────────────
def _get_copt_constant(names: List[str], default: int = 0) -> int:
    for name in names:
        if hasattr(COPT, name):
            return getattr(COPT, name)
    return default


CTX_MIPSOL = _get_copt_constant(["CBC_MIP_SOL", "CB_MIPSOL", "MIPSOL"], 0)
CTX_MIPNODE = _get_copt_constant(["CBC_MIP_NODE", "CB_MIPNODE", "MIPNODE"], 0)

if CTX_MIPSOL == 0 and CTX_MIPNODE == 0:
    print(
        "Warning: Could not detect COPT MIP callback context constants. "
        "This is harmless in v94.21 (no callbacks used)."
    )

# ─────────────────────────────────────────────────────────────────────
# PATHS (UPDATED FOR DUMMY FILE)
# ─────────────────────────────────────────────────────────────────────
ROOT = Path(r"C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation")
DATA_FILE = ROOT / "dummy_cost_benefits.xlsx"

CACHE = ROOT / "scenario_cache_dummy_benefit_mo"
CACHE.mkdir(exist_ok=True)

PKL_PREFIX: str | None = "dummy_"

# ─────────────────────────────────────────────────────────────────────
# CALENDAR / ECON (UPDATED FOR 40-YEAR HORIZON)
# ─────────────────────────────────────────────────────────────────────
START_FY = 2026

EVALUATION_YEARS = 40  # <-- requested
FINAL_YEAR = START_FY + EVALUATION_YEARS - 1  # 2026..2065 inclusive

TFIXED = int(FINAL_YEAR - START_FY + 1)
YEARS = TFIXED

BENEFIT_DISCOUNT_RATE = 0.02

# Scaling: 1 unit of spend in the model = 0.01 M (10k)
SPEND_SCALE = 100       # 1 unit = 0.01 M
PV_SCALE = 10000        # integer PV scaling (kept to preserve your objective behavior)

MAX_STARTS_PER_FY = 100  # Max project starts allowed in a single year

# ─────────────────────────────────────────────────────────────────────
# FUNDING ENVELOPE CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
TAPER_YEARS = 0

# ─────────────────────────────────────────────────────────────────────
# PIECEWISE SOFT CAP SETTINGS (The "Tax Brackets")
# ─────────────────────────────────────────────────────────────────────
PIECEWISE_CAP_TIERS: List[Tuple[float, float]] = [
    (0.12, 1000.0),
    (0.15, 4000.0),
    (0.20, 12000.0),
]

# ─────────────────────────────────────────────────────────────────────
# OBJECTIVE WEIGHTS
# ─────────────────────────────────────────────────────────────────────
BACKLOG_WEIGHT = 1.0            # Penalise normal net balance
PV_WEIGHT = 1e-4                # PV tie-breaker (NOTE: with PV_SCALE, effective weight is PV_WEIGHT*PV_SCALE)

# ─────────────────────────────────────────────────────────────────────
# SOLVER GLOBALS
# ─────────────────────────────────────────────────────────────────────
SOLVER_SEED_DEFAULT = 17
VERBOSE = 2

# Profile grid (time in seconds, target relative gap).
OPTIMISATION_PROFILE = "ultra"  # "fast", "balanced", "thorough", "ultra"

# IMPORTANT: RelGap is a FRACTION. 0.01% => 0.0001
EFFORT: Dict[str, Dict[str, float]] = {
    "fast":     {"MO": 60.0,  "REL_GAP": 0.0001},
    "balanced": {"MO": 120.0, "REL_GAP": 0.0001},
    "thorough": {"MO": 360.0, "REL_GAP": 0.0001},
    "ultra":    {"MO": 900.0, "REL_GAP": 0.0001},
}

VALIDATE_SOLUTION = True
VALIDATION_TOL_M = 1e-3

# ─────────────────────────────────────────────────────────────────────
# RUN PROFILE (UPDATED BENEFITS SHEET NAME)
# ─────────────────────────────────────────────────────────────────────
COST_TYPES_RUN: List[str] = ["P50 - Real"]
BENEFIT_SCENARIOS: Dict[str, str] = {"DUMMY": "Benefits"}  # <-- dummy workbook sheet

SURPLUS_OPTIONS_M: Dict[str, float] = {"s500": 500.0, "s1000": 1000.0,}
PLUSMINUS_LEVELS_M: List[float] = [0.0, 250.0]

# --- PROJECT CONSTRAINTS -----------------------------------------------------
# Force-start: keep ALL projects (unless explicitly excluded), but force these starts / includes.
FORCED_START: Dict[str, Dict] = {
   # "Project 217": {"start": 2026, "include": True},
}

# Earliest allowed start year (applies unless project is forced to an exact start year)
MIN_START_YEAR: Dict[str, int] = {
    # "Project 217": 2026,
}

# ---------------------------
# SELECTION MODE:
#   - "auto": keep all projects except exclude=True, and apply include/start directives
#   - "blacklist": same as auto but explicitly blacklist semantics
#   - "whitelist": keep ONLY include=True projects (unless fallback enabled)
PROJECT_SELECTION_MODE = "auto"
WHITELIST_FALLBACK_TO_BLACKLIST_IF_EMPTY = True
WARN_ON_UNMATCHED_RULE_NAMES = True

# UPDATED to match your dummy file (only these + Total exist)
DIMENSION_INCLUSIONS: Dict[str, bool] = {
    "Total": True,
    "Economic Prosperity": True,
    "Healthy and safe people": True,
    "Inclusive Access": True,
}

# ─────────────────────────────────────────────────────────────────────
# MONOTONE PV GUARD BETWEEN BUFFERS
# ─────────────────────────────────────────────────────────────────────
ENFORCE_MONOTONE_PV_ACROSS_BUFFERS = True
MONO_REL_EPS = 1e-4
MONO_ABS_EPS = 1e-3

# ─────────────────────────────────────────────────────────────────────
# FORMULATION / PERFORMANCE SWITCHES
# ─────────────────────────────────────────────────────────────────────
USE_INDICATOR_CONSTRAINTS = True  # Stronger than big-M when y is binary.
MULTI_START_COUNT = 6             # How many diversified starts to generate.
LOCAL_IMPROVE_PASSES = 1          # Local-search passes over projects for warm starts.
LOCAL_IMPROVE_WINDOW = 2          # Try shifting each project by +/- this many years.

# ─────────────────────────────────────────────────────────────────────
# LOGGING / HELPERS
# ─────────────────────────────────────────────────────────────────────
def cal_years(ny: int) -> List[int]:
    return [START_FY + i for i in range(ny)]


def clean(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "").replace("\xa0", " ")).strip()


def norm(s: str) -> str:
    return clean(s).lower()


def iround(x: float, scale: float) -> int:
    return int(round(float(x) * float(scale)))


def _log(msg: str) -> None:
    if VERBOSE >= 1:
        print(msg)


def _warn(msg: str) -> None:
    if VERBOSE >= 2:
        print("Warning:", msg)


def _now_stamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S", time.localtime())


# Robust numeric parsing (handles commas, dashes, parentheses, etc.)
_DASH_LITERALS = {"-", "–", "—", " - ", " -- ", "--", "—-", "–-", "n/a", "na", "null", "none", ""}


def to_float(x: Any) -> float:
    if x is None:
        return 0.0
    if isinstance(x, (int, float, np.integer, np.floating)):
        try:
            if float(x) != float(x):  # NaN
                return 0.0
        except Exception:
            return 0.0
        return float(x)

    s = clean(str(x))
    if norm(s) in _DASH_LITERALS:
        return 0.0

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1].strip()

    # Remove currency symbols and thousands separators; keep exponent notation.
    s = s.replace(",", "")
    s = re.sub(r"[^0-9eE\.\-\+]", "", s).strip()

    if s in {"", "-", "+", ".", "-.", "+."}:
        return 0.0

    try:
        v = float(s)
    except Exception:
        return 0.0
    return -v if neg else v


class ParamChangeLogger:
    def __init__(self):
        self._last_params: Dict[str, Any] = {}

    def log_changes(self, params: Dict[str, Any], header: str = "Param changes") -> None:
        changed = {
            k: v
            for k, v in params.items()
            if k not in self._last_params or self._last_params[k] != v
        }
        if changed:
            self._last_params.update(params)
            changed_str = ", ".join(f"{k}={repr(v)}" for k, v in sorted(changed.items()))
            print(f"{header}: {changed_str}")


_PARAM_LOGGER = ParamChangeLogger()


def _apply_params_logged(m: co.Model, updates: Dict[str, Any], header: str) -> None:
    _PARAM_LOGGER.log_changes(updates, header=header)
    for k, v in updates.items():
        try:
            m.setParam(k, v)
        except Exception as e:
            _warn(f"Failed to setParam({k}={v!r}): {e}")


def _log_effective_parallelism(m: co.Model, where: str) -> None:
    try:
        thr = m.getParam("Threads")
    except Exception:
        thr = None
    try:
        tasks = m.getParam("MipTasks")
    except Exception:
        tasks = None
    if VERBOSE >= 1:
        print(f"[parallel] {where}: Threads={thr!r}, MipTasks={tasks!r}")


def _has_incumbent(m: co.Model) -> bool:
    try:
        hm = m.getAttr(COPT.Attr.HasMipSol)
        if hm is not None:
            return bool(hm)
    except Exception:
        pass
    try:
        m.getAttr(COPT.Attr.ObjVal)
        return True
    except Exception:
        return False


def _best_gap(m: co.Model) -> Optional[float]:
    try:
        g = m.getAttr(COPT.Attr.BestGap)
        if g is not None:
            return float(g)
    except Exception:
        pass
    try:
        obj = float(m.getAttr(COPT.Attr.ObjVal))
        bnd = float(m.getAttr(COPT.Attr.BestBnd))
        denom = max(1.0, abs(obj))
        return abs(bnd - obj) / denom
    except Exception:
        return None


RUN_ID = hashlib.sha1(
    f"{OPTIMISATION_PROFILE}|{SOLVER_SEED_DEFAULT}|{SOLVER_THREADS_DEFAULT}|{SOLVER_TASKS_DEFAULT}".encode("utf-8")
).hexdigest()[:8]


def _val(val_by_id: Optional[Dict[int, float]], var: "co.Var") -> float:
    try:
        if val_by_id is not None:
            v = val_by_id.get(id(var), None)
            if v is not None:
                return float(v)
        return float(var.X)
    except Exception:
        return 0.0


def pkl_save(path: Path, payload: Dict[str, Any]) -> None:
    with path.open("wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    size = path.stat().st_size if path.exists() else 0
    objective = payload.get("objective", payload.get("primary_dim", "?"))
    best = payload.get("best", {}) or {}
    pv = best.get("pv", payload.get("pv_total", "?"))

    gap_frac = best.get("gap", payload.get("gap", None))
    gap_pct_val = best.get("gap_pct", payload.get("gap_pct", None))

    if isinstance(gap_pct_val, (int, float)):
        gap_str = f"{gap_pct_val:.4f}%"
    elif isinstance(gap_frac, (int, float)):
        gap_str = f"{gap_frac * 100.0:.4f}%"
    else:
        gap_str = "?"

    print(f"Saved: {path} ({size} bytes) objective={objective} pv={pv} gap={gap_str}")


# ─────────────────────────────────────────────────────────────────────
# I/O – Costs & Benefits (UPDATED FOR DUMMY SHEETS / FORMATS)
# ─────────────────────────────────────────────────────────────────────
def _col_lookup(df: pd.DataFrame, want_norm: str) -> Optional[str]:
    want_norm = norm(want_norm)
    for c in df.columns:
        if norm(c) == want_norm:
            return c
    return None


def load_costs(cost_type: str):
    """
    Dummy Costs sheet format supported:
      Project | Cost type | ... | Duration | 2026 | 2027 | ...
    Robust to values like "106,839,620.21" and " - " placeholders.

    Returns:
      projects, variants, costs_input_df
    """
    df = pd.read_excel(DATA_FILE, sheet_name="Costs", engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    proj_col = _col_lookup(df, "Project")
    if not proj_col:
        raise RuntimeError("Costs sheet needs a 'Project' column.")

    cost_type_col = _col_lookup(df, "Cost type")
    if not cost_type_col:
        raise RuntimeError("Costs sheet needs a 'Cost type' column.")

    duration_col = _col_lookup(df, "Duration")  # optional, but present in your dummy file
    # cost_total_col = _col_lookup(df, "Cost")  # optional, not used by model

    horizon_all = [START_FY + i for i in range(YEARS)]
    year_cols = {int(c): c for c in df.columns if str(c).isdigit()}
    use_cols = [year_cols.get(y, None) for y in horizon_all]

    cut = df[df[cost_type_col].astype(str).map(clean) == str(cost_type).strip()].copy()

    # Aggregate per-project in case the sheet has multiple rows per project (robust)
    costs_by_project = {}
    durations_by_project: Dict[str, List[int]] = defaultdict(list)

    for _, r in cut.iterrows():
        p = clean(r[proj_col])
        if not p:
            continue

        seq = np.zeros(len(horizon_all), dtype=float)
        for i, c in enumerate(use_cols):
            if c is None:
                seq[i] = 0.0
            else:
                seq[i] = to_float(r.get(c, 0.0))

        if p in costs_by_project:
            costs_by_project[p] = costs_by_project[p] + seq
        else:
            costs_by_project[p] = seq

        if duration_col:
            d = int(round(to_float(r.get(duration_col, 0.0))))
            if d > 0:
                durations_by_project[p].append(d)

    # Convert to M and build inputs
    costs_input: Dict[str, List[float]] = {}
    for p, seq_dollars in costs_by_project.items():
        costs_input[p] = (seq_dollars / 1_000_000.0).tolist()  # M

    projects: Dict[str, Dict[str, Any]] = {}
    variants: Dict[str, Dict[str, Any]] = {}

    for p, seriesM in costs_input.items():
        s = pd.Series(seriesM, dtype=float)
        nz = np.where(np.abs(s.to_numpy(dtype=float)) > 1e-12)[0]
        if nz.size == 0:
            continue

        first_idx = int(nz.min())

        dur_from_col: Optional[int] = None
        dlist = durations_by_project.get(p, [])
        if dlist:
            # Choose the most common duration (mode). If tie, choose max.
            counts = defaultdict(int)
            for d in dlist:
                counts[int(d)] += 1
            dur_from_col = sorted(counts.keys(), key=lambda k: (counts[k], k), reverse=True)[0]

        if dur_from_col is not None and dur_from_col > 0:
            dur = int(dur_from_col)
            seg = s.iloc[first_idx:first_idx + dur].tolist()
            if len(seg) < dur:
                seg = seg + [0.0] * (dur - len(seg))
        else:
            last_idx = int(nz.max())
            seg = s.iloc[first_idx:last_idx + 1].tolist()
            dur = len(seg)

        projects[p] = {"cost": float(sum(seg)), "dur": int(dur), "spend": [float(x) for x in seg]}
        variants[p] = {
            "base": p,
            "dur": int(dur),
            "spend": [float(x) for x in seg],
            "first_year_idx": int(first_idx),
        }

    costs_input_df = pd.DataFrame(costs_input, index=horizon_all).T
    costs_input_df.index.name = "Project"
    return projects, variants, costs_input_df


def load_benefits(sheet: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Dummy Benefits sheet format supported:
      Project | ... | Dimension | t+0 | t+1 | ... | t+40
    Extra descriptor columns are ignored.
    """
    df = pd.read_excel(DATA_FILE, sheet_name=sheet, engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    proj_col = _col_lookup(df, "Project")
    if not proj_col:
        raise RuntimeError("Benefits sheet needs a 'Project' column.")
    if proj_col != "Project":
        df.rename(columns={proj_col: "Project"}, inplace=True)

    # Dimension column can sometimes have weird casing/spacing
    dim_col = None
    for c in df.columns:
        if norm(c).startswith("dimension"):
            dim_col = c
            break
    if dim_col is None:
        raise RuntimeError("Benefits sheet needs a 'Dimension' column (case-insensitive match).")
    if dim_col != "Dimension":
        df.rename(columns={dim_col: "Dimension"}, inplace=True)

    # Collect t+ columns (robust to spaces/casing)
    tcols: List[Tuple[int, str]] = []
    for c in df.columns:
        m = re.fullmatch(r"[tT]\s*\+\s*(\d+)", str(c))
        if m:
            tcols.append((int(m.group(1)), c))
    tcols.sort(key=lambda x: x[0])

    if not tcols:
        raise RuntimeError("Benefits sheet has no columns matching 't+N' (e.g., t+0..t+40).")

    # Robust numeric conversion for benefit columns
    for _, c in tcols:
        df[c] = df[c].map(to_float).astype(float)

    ben_kernel_df = df.copy()
    ben_kernel_df["Project"] = ben_kernel_df["Project"].map(clean)
    ben_kernel_df["Dimension"] = ben_kernel_df["Dimension"].map(clean)
    ben_kernel_df.set_index(["Project", "Dimension"], inplace=True)
    ben_kernel_df = ben_kernel_df[[c for _, c in tcols]]
    return df, ben_kernel_df


def map_benefit_kernels(benef_df: pd.DataFrame, variants: Dict[str, dict]):
    """
    Builds per-dimension, per-project benefit kernels.

    Robust behaviour vs dummy file quirks:
      - If multiple rows exist for the same (Project, Dimension), their t+ streams are SUMMED.
      - If t+ indices are not contiguous, missing years are filled with 0.0.
      - Only projects present in `variants` are kept.
      - Benefits are shifted by +dur years (construction), i.e. [0]*dur + [t+0..]
    """
    # Parse t+ columns with indices
    t_pairs: List[Tuple[int, str]] = []
    for c in benef_df.columns:
        m = re.fullmatch(r"[tT]\s*\+\s*(\d+)", str(c))
        if m:
            t_pairs.append((int(m.group(1)), c))
    t_pairs.sort(key=lambda x: x[0])
    if not t_pairs:
        raise RuntimeError("Benefits dataframe contains no 't+N' columns.")

    tmax = int(t_pairs[-1][0])
    # Ensure we can build full vectors of length tmax+1, filling gaps with zeros
    df = benef_df.copy()
    df["Project_clean"] = df["Project"].map(clean)
    df["Dimension_clean"] = df["Dimension"].map(clean)

    flows_by_dim: Dict[str, Dict[str, List[float]]] = {}
    order: List[str] = []

    for _, r in df.iterrows():
        p = r["Project_clean"]
        d = r["Dimension_clean"]
        if not p or not d:
            continue

        seq_full = [0.0] * (tmax + 1)
        for idx, col in t_pairs:
            seq_full[idx] = float(to_float(r.get(col, 0.0)))

        flows_by_dim.setdefault(d, {})
        if p in flows_by_dim[d]:
            prev = flows_by_dim[d][p]
            flows_by_dim[d][p] = [a + b for a, b in zip(prev, seq_full)]
        else:
            flows_by_dim[d][p] = seq_full

        if d not in order:
            order.append(d)

    # If Total missing, compute it as sum of other dims (excluding Total)
    if "Total" not in flows_by_dim:
        flows_by_dim["Total"] = {}

    all_dims = [d for d in order if d.lower() != "total"]
    projs = set()
    for d in all_dims:
        projs |= set(flows_by_dim[d].keys())

    if "Total" not in order:
        order.append("Total")

    if flows_by_dim.get("Total") is not None and len(flows_by_dim["Total"]) == 0:
        for p in projs:
            acc = [0.0] * (tmax + 1)
            for d in all_dims:
                v = flows_by_dim[d].get(p)
                if v is None:
                    continue
                acc = [a + b for a, b in zip(acc, v)]
            flows_by_dim["Total"][p] = acc

    # Keep only projects we have cost variants for
    keeps = set(variants.keys())
    for d in list(flows_by_dim.keys()):
        flows_by_dim[d] = {p: seq for p, seq in flows_by_dim[d].items() if p in keeps}

    # Build kernels shifted by +dur
    kernels_by_dim: Dict[str, Dict[str, List[float]]] = {}
    for d, mp_ in flows_by_dim.items():
        kernels_by_dim[d] = {}
        for v, meta in variants.items():
            dur = int(meta["dur"])
            ker = mp_.get(v, [])
            kernels_by_dim[d][v] = [0.0] * dur + [float(x) for x in ker]

    return order, kernels_by_dim


# ─────────────────────────────────────────────────────────────────────
# Rules / Starts  (FIXED: auto mode no longer turns into whitelist)
# ─────────────────────────────────────────────────────────────────────
def apply_forced_rules(variants: Dict[str, dict], rules: Dict[str, Dict]):
    """
    Returns:
      kept_variants: filtered variants dict
      forced_exact: dict {project_name: start_year} for projects with forced start year
      is_whitelist_model: True only if PROJECT_SELECTION_MODE == "whitelist" and whitelist is in effect

    FIX (v94.21):
      - "auto" no longer becomes WHITELIST just because include=True rules exist.
      - default is keep-all-minus-excludes (blacklist-style) and apply include/start directives.
    """
    v_norm2canon = {norm(v): v for v in variants.keys()}
    v_norm_set = set(v_norm2canon.keys())

    include_true_norm, exclude_true_norm = set(), set()
    start_map_all_norm: Dict[str, int] = {}

    unmatched_rule_names: List[str] = []

    for raw_name, spec in (rules or {}).items():
        pname_norm = norm(raw_name)
        inc = spec.get("include", None)
        st = spec.get("start", None)

        if pname_norm not in v_norm_set and WARN_ON_UNMATCHED_RULE_NAMES:
            unmatched_rule_names.append(raw_name)

        if inc is True:
            include_true_norm.add(pname_norm)
        if inc is False:
            exclude_true_norm.add(pname_norm)
        if st is not None:
            try:
                start_map_all_norm[pname_norm] = int(st)
            except Exception:
                pass

    if unmatched_rule_names and WARN_ON_UNMATCHED_RULE_NAMES and VERBOSE >= 2:
        _warn(f"[rules] Unmatched rule names (not found in variants): {unmatched_rule_names[:10]}")

    matched_includes_norm = include_true_norm & v_norm_set
    matched_excludes_norm = exclude_true_norm & v_norm_set

    mode_req = (PROJECT_SELECTION_MODE or "auto").strip().lower()

    # include=True overrides exclude=True when both present
    effective_excludes = matched_excludes_norm - matched_includes_norm

    # Only explicit whitelist triggers whitelist.
    use_whitelist = (mode_req == "whitelist")

    if use_whitelist:
        keep_norm = set(matched_includes_norm)
        mode = "WHITELIST"
        if len(keep_norm) == 0 and WHITELIST_FALLBACK_TO_BLACKLIST_IF_EMPTY:
            keep_norm = set(v_norm_set) - effective_excludes
            mode = "BLACKLIST (fallback from empty whitelist)"
            use_whitelist = False
    else:
        # "auto" and "blacklist" both behave as blacklist-by-default
        keep_norm = set(v_norm_set) - effective_excludes
        # ensure includes kept (already true, but explicit)
        keep_norm |= matched_includes_norm
        mode = "BLACKLIST (auto)" if mode_req == "auto" else "BLACKLIST"

    keep_canon = {v_norm2canon[n] for n in keep_norm}
    kept_variants = {v: variants[v] for v in variants if v in keep_canon}

    forced_exact: Dict[str, int] = {}
    for n, yr in start_map_all_norm.items():
        if n in keep_norm:
            forced_exact[v_norm2canon[n]] = int(yr)

    if VERBOSE >= 1:
        print(f" [rules] Mode={mode}; kept={len(kept_variants)}/{len(variants)}; forced={len(forced_exact)}")

    return kept_variants, forced_exact, bool(use_whitelist)


def allowed_starts_fine(
    variants: Dict[str, dict],
    forced_exact: Dict[str, int],
    min_starts: Dict[str, int],
    Tfine: int,
) -> Dict[str, List[int]]:
    ny = Tfine
    allowed: Dict[str, List[int]] = {}
    for v, meta in variants.items():
        dur = meta["dur"]

        # Forced exact start overrides min-start logic
        if v in forced_exact and forced_exact[v] is not None:
            s = forced_exact[v] - START_FY
            allowed[v] = [s] if (0 <= s <= ny - dur) else []
            continue

        s_ear = 0
        s_lat = ny - dur

        min_yr = min_starts.get(v)
        if min_yr is not None:
            s_min_constraint = min_yr - START_FY
            s_ear = max(s_ear, s_min_constraint)

        allowed[v] = list(range(s_ear, s_lat + 1)) if s_lat >= s_ear else []
    return allowed


# ─────────────────────────────────────────────────────────────────────
# PV Coeff maps
# ─────────────────────────────────────────────────────────────────────
def coeff_map_for_dim_fine(
    variants: Dict[str, dict],
    kernels_for_dim: Dict[str, List[float]],
    allowed: Dict[str, List[int]],
    Tfine: int,
    disc_vec: np.ndarray,
) -> Dict[Tuple[str, int], float]:
    out: Dict[Tuple[str, int], float] = {}
    for v in variants.keys():
        dur = variants[v]["dur"]
        ker = kernels_for_dim.get(v, [])
        if not ker:
            continue
        for s in allowed.get(v, []):
            val = 0.0
            for k, f in enumerate(ker):
                if f == 0.0:
                    continue
                t = s + k
                if 0 <= t < Tfine:
                    val += float(f) / float(disc_vec[t])
            if val != 0.0:
                out[(v, s)] = val
    return out


def coeff_int(coeff_map: Dict[Tuple[str, int], float], scale: float = PV_SCALE) -> Dict[Tuple[str, int], int]:
    return {k: int(round(v * scale)) for k, v in coeff_map.items() if v != 0.0}


# ─────────────────────────────────────────────────────────────────────
# Warm start generation (multi-start + local polish)
# ─────────────────────────────────────────────────────────────────────
def _selection_to_start_map(sel: Dict[Tuple[str, int], int]) -> Dict[str, int]:
    out: Dict[str, int] = {}
    for (v, s), on in sel.items():
        if on:
            out[v] = int(s)
    return out


def _start_map_to_selection(start_map: Dict[str, int]) -> Dict[Tuple[str, int], int]:
    return {(v, int(s)): 1 for v, s in start_map.items()}


def _compute_spend_series_M(
    variants: Dict[str, dict],
    start_map: Dict[str, int],
    Tfine: int
) -> np.ndarray:
    spend = np.zeros(Tfine, dtype=float)
    for v, s in start_map.items():
        vec = variants[v]["spend"]
        for k, amt in enumerate(vec):
            t = s + k
            if 0 <= t < Tfine and amt != 0.0:
                spend[t] += float(amt)
    return spend


def _piecewise_penalty_from_net_S(net_S: float, env_S: float) -> float:
    base_threshold = PIECEWISE_CAP_TIERS[0][0]
    base_cap_S = env_S * base_threshold
    excess = max(0.0, net_S - base_cap_S)
    if excess <= 0.0:
        return 0.0

    pen = 0.0
    for i, (th_start, weight) in enumerate(PIECEWISE_CAP_TIERS):
        is_last = i == len(PIECEWISE_CAP_TIERS) - 1
        if not is_last:
            th_next = PIECEWISE_CAP_TIERS[i + 1][0]
            width_S = env_S * (th_next - th_start)
            take = min(excess, width_S)
            pen += take * float(weight)
            excess -= take
            if excess <= 1e-12:
                break
        else:
            pen += excess * float(weight)
            break
    return float(pen)


def _approx_objective_for_start_map(
    variants: Dict[str, dict],
    start_map: Dict[str, int],
    Tfine: int,
    funding_target_M: np.ndarray,
    coeff_total_fine: Dict[Tuple[str, int], float],
) -> float:
    spend_M = _compute_spend_series_M(variants, start_map, Tfine)

    nz = np.nonzero(spend_M > 1e-12)[0]
    if nz.size == 0:
        return float("inf")
    L = int(nz.max())  # last spend year index

    env_S_by_t = np.array([iround(float(m), SPEND_SCALE) for m in funding_target_M[:Tfine]], dtype=float)
    spend_S = spend_M * float(SPEND_SCALE)

    net_S = np.zeros(Tfine, dtype=float)
    for t in range(Tfine):
        fund_S = env_S_by_t[t] if t <= L else 0.0
        net_S[t] = (fund_S - spend_S[t]) if t == 0 else (net_S[t - 1] + fund_S - spend_S[t])
        if net_S[t] < -1e-9:
            return float("inf")  # violates no-debt

    backlog_sum_S = float(np.sum(net_S[:max(0, L)]))
    pen = 0.0
    for t in range(max(0, L)):
        pen += _piecewise_penalty_from_net_S(net_S[t], env_S_by_t[t])

    pv = 0.0
    for v, s in start_map.items():
        pv += float(coeff_total_fine.get((v, int(s)), 0.0))
    pv_term = float(PV_WEIGHT) * float(PV_SCALE) * pv

    return float(BACKLOG_WEIGHT) * backlog_sum_S + float(pen) - pv_term


def greedy_warm_start(
    variants: Dict[str, dict],
    allowed: Dict[str, List[int]],
    Tfine: int,
    funding_target_M: np.ndarray,
    max_starts_per_year: int,
    forced_exact: Dict[str, int],
    coeff_total_map: Dict[Tuple[str, int], float],
    reuse_sel: Optional[Dict[Tuple[str, int], int]] = None,
    *,
    mode: str = "pv_per_cost",
    rng: Optional[random.Random] = None,
) -> Dict[Tuple[str, int], int]:
    rng = rng or random.Random(0)

    ny = Tfine
    capacity_prefix = np.cumsum(funding_target_M[:ny])
    spend_cum = np.zeros(ny, dtype=float)
    starts_count = np.zeros(ny, dtype=int)
    sel: Dict[Tuple[str, int], int] = {}

    # Try to reuse previous selection if it fits
    if reuse_sel:
        ok = True
        tmp_sel: Dict[Tuple[str, int], int] = {}
        tmp_count = starts_count.copy()
        tmp_cum = spend_cum.copy()
        for (v, s), on in sorted(reuse_sel.items(), key=lambda kv: (kv[0][1], kv[0][0])):
            if not on:
                continue
            if s not in set(allowed.get(v, [])):
                ok = False
                break
            if v not in variants:
                ok = False
                break
            if tmp_count[s] >= max_starts_per_year:
                ok = False
                break
            d = variants[v]["dur"]
            vec = np.array(variants[v]["spend"], dtype=float)
            inc = np.zeros(ny, dtype=float)
            inc[s: s + d] = vec
            if np.any(tmp_cum + np.cumsum(inc) - capacity_prefix > 1e-9):
                ok = False
                break
            tmp_cum += np.cumsum(inc)
            tmp_count[s] += 1
            tmp_sel[(v, s)] = 1
        if ok:
            sel = tmp_sel
            starts_count = tmp_count
            spend_cum = tmp_cum

    # Enforce forced projects
    forced_order: List[Tuple[int, str]] = []
    for v, yr in (forced_exact or {}).items():
        if yr is None or v not in variants:
            continue
        s = int(yr - START_FY)
        d = variants[v]["dur"]
        if s < 0 or s > ny - d:
            continue
        if s not in set(allowed.get(v, [])):
            continue
        if (v, s) not in sel:
            forced_order.append((s, v))
    forced_order.sort()

    for s, v in forced_order:
        if starts_count[s] >= max_starts_per_year:
            continue
        d = variants[v]["dur"]
        vec = np.array(variants[v]["spend"], dtype=float)
        inc = np.zeros(ny, dtype=float)
        inc[s: s + d] = vec
        if np.any(spend_cum + np.cumsum(inc) - capacity_prefix > 1e-9):
            continue
        spend_cum += np.cumsum(inc)
        starts_count[s] += 1
        sel[(v, s)] = 1

    already = {vv for (vv, _) in sel.keys()}
    remain = [v for v in variants.keys() if v not in already]

    scored: List[Tuple[float, str]] = []
    for v in remain:
        denom = float(sum(variants[v]["spend"])) or 1e-9
        bestpv = max((coeff_total_map.get((v, s), 0.0) for s in allowed.get(v, [])), default=0.0)

        if mode == "pv":
            score = bestpv
        elif mode == "early_spend":
            best_s = None
            best_s_pv = -1e100
            for s in allowed.get(v, []):
                pv_s = coeff_total_map.get((v, s), 0.0)
                if pv_s > best_s_pv:
                    best_s_pv = pv_s
                    best_s = s
            if best_s is None:
                score = bestpv / denom
            else:
                score = (bestpv / denom) * (1.0 / (1.0 + 0.1 * float(best_s)))
        else:
            score = bestpv / denom

        score *= (1.0 + 0.001 * (rng.random() - 0.5))
        scored.append((float(score), v))

    scored.sort(reverse=True)

    for _score, v in scored:
        d = variants[v]["dur"]
        vec = np.array(variants[v]["spend"], dtype=float)

        pv_starts: List[Tuple[float, int]] = []
        for s in allowed.get(v, []):
            pv_starts.append((float(coeff_total_map.get((v, s), 0.0)) * (1.0 + 0.0005 * (rng.random() - 0.5)), int(s)))
        pv_starts.sort(key=lambda x: x[0], reverse=True)

        for _pv, s in pv_starts:
            if starts_count[s] >= max_starts_per_year:
                continue
            inc = np.zeros(ny, dtype=float)
            inc[s: s + d] = vec
            if np.any(spend_cum + np.cumsum(inc) - capacity_prefix > 1e-9):
                continue
            spend_cum += np.cumsum(inc)
            starts_count[s] += 1
            sel[(v, s)] = 1
            break

    return sel


def _local_improve_start_map(
    variants: Dict[str, dict],
    allowed: Dict[str, List[int]],
    start_map: Dict[str, int],
    Tfine: int,
    funding_target_M: np.ndarray,
    max_starts_per_year: int,
    coeff_total_fine: Dict[Tuple[str, int], float],
    *,
    passes: int = 1,
    window: int = 2,
) -> Dict[str, int]:
    proj_list = list(start_map.keys())
    funding_target_M = funding_target_M[:Tfine]

    def feasible(sm: Dict[str, int]) -> bool:
        counts = np.zeros(Tfine, dtype=int)
        for v, s in sm.items():
            if s < 0 or s >= Tfine:
                return False
            counts[s] += 1
            if counts[s] > max_starts_per_year:
                return False

        spend = _compute_spend_series_M(variants, sm, Tfine)
        cum_spend = np.cumsum(spend)
        cum_fund = np.cumsum(funding_target_M)
        return bool(np.all(cum_spend - cum_fund <= 1e-9))

    best = dict(start_map)
    best_obj = _approx_objective_for_start_map(variants, best, Tfine, funding_target_M, coeff_total_fine)

    for _ in range(max(1, passes)):
        improved = False
        for v in proj_list:
            cur_s = int(best[v])
            cand_s_list = [cur_s + d for d in range(-window, window + 1) if d != 0]
            allowed_set = set(allowed.get(v, []))
            cand_s_list = [s for s in cand_s_list if s in allowed_set]
            if not cand_s_list:
                continue

            for s2 in cand_s_list:
                trial = dict(best)
                trial[v] = int(s2)
                if not feasible(trial):
                    continue
                obj = _approx_objective_for_start_map(variants, trial, Tfine, funding_target_M, coeff_total_fine)
                if obj < best_obj:
                    best = trial
                    best_obj = obj
                    improved = True
        if not improved:
            break

    return best


def build_diversified_mip_starts(
    variants: Dict[str, dict],
    allowed: Dict[str, List[int]],
    Tfine: int,
    funding_target_M: np.ndarray,
    max_starts_per_year: int,
    forced_exact: Dict[str, int],
    coeff_total_fine: Dict[Tuple[str, int], float],
    reuse_sel: Optional[Dict[Tuple[str, int], int]],
    *,
    n_starts: int,
) -> List[Dict[Tuple[str, int], int]]:
    modes = ["pv_per_cost", "pv", "early_spend"]
    starts: List[Dict[Tuple[str, int], int]] = []

    for i in range(max(1, n_starts)):
        mode = modes[i % len(modes)]
        rlocal = random.Random(SOLVER_SEED_DEFAULT + 1000 + i)
        base_sel = greedy_warm_start(
            variants,
            allowed,
            Tfine,
            funding_target_M,
            max_starts_per_year,
            forced_exact,
            coeff_total_fine,
            reuse_sel=reuse_sel if i == 0 else None,
            mode=mode,
            rng=rlocal,
        )
        sm = _selection_to_start_map(base_sel)

        if LOCAL_IMPROVE_PASSES > 0 and LOCAL_IMPROVE_WINDOW > 0 and sm:
            sm2 = _local_improve_start_map(
                variants,
                allowed,
                sm,
                Tfine,
                funding_target_M,
                max_starts_per_year,
                coeff_total_fine,
                passes=LOCAL_IMPROVE_PASSES,
                window=LOCAL_IMPROVE_WINDOW,
            )
            base_sel = _start_map_to_selection(sm2)

        if base_sel:
            starts.append(base_sel)

    uniq: Dict[str, Dict[Tuple[str, int], int]] = {}
    for sel in starts:
        h = hashlib.sha1(
            json.dumps(sorted([(v, s) for (v, s), on in sel.items() if on]), separators=(",", ":")).encode("utf-8")
        ).hexdigest()
        uniq.setdefault(h, sel)

    return list(uniq.values())


# ─────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────
class CoptSpendMatchMO:
    """
    Spend / funding model with:
      - funding[t] FIXED to envelope[t] if active.
      - PIECEWISE SOFT CAP on net[t] (tiers of penalties).
      - objective = minimize backlog + penalties - PV reward
      - MUST PURCHASE EVERY PROJECT (Hard start constraint = 1.0)
      - DIVIDEND[t]: Restricted to end-of-life (when y[t+1] = 0).
    """

    def __init__(
        self,
        variants: Dict[str, dict],
        allowed: Dict[str, List[int]],
        funding_target_S: np.ndarray,  # scaled per-year capacity
        Tn: int,
        taper_start_idx: int,
        spend_by_year: Dict[str, List[float]],
        max_starts_per_year: int,
        is_whitelist_model: bool,
        env: Optional[co.Envr] = None,
        name: str = "spend_match_mo",
        *,
        relax_binaries: bool = False,
        use_indicators: bool = True,
    ):
        self.Tn = int(Tn)
        self.taper_start_idx = taper_start_idx

        self.env = env or co.Envr()
        self.m: co.Model = self.env.createModel(name)

        init_params = {
            "RandSeed": int(SOLVER_SEED_DEFAULT),
            "Threads": int(max(1, SOLVER_THREADS_DEFAULT)),
            "MipTasks": int(max(1, SOLVER_TASKS_DEFAULT)),
            "MipStartMode": 2,
            "Logging": 1,
            "Presolve": 1,
            "Scaling": 1,
        }
        _apply_params_logged(self.m, init_params, header="Model init params")
        _log_effective_parallelism(self.m, where=f"{name}/init")

        self.use_indicators = bool(use_indicators and (not relax_binaries) and hasattr(self.m, "addGenConstrIndicator"))

        self.funding_target_S = funding_target_S

        # Upper bounds for net based on cumulative funding
        self.net_ub: List[float] = []
        cumulative_fund = 0.0
        for t in range(self.Tn):
            cumulative_fund += float(self.funding_target_S[t])
            self.net_ub.append(cumulative_fund)

        # Decision x[v,s]
        self.x: Dict[Tuple[str, int], co.Var] = {}
        self.by_year: Dict[int, List[Tuple[str, int]]] = defaultdict(list)
        x_vtype = COPT.CONTINUOUS if relax_binaries else COPT.BINARY
        for v in variants.keys():
            for s in allowed.get(v, []):
                var = self.m.addVar(lb=0.0, ub=1.0, vtype=x_vtype, name=f"x[{v}|{s}]")
                self.x[(v, s)] = var
                if 0 <= s < self.Tn:
                    self.by_year[s].append((v, s))

        # Envelope-active indicator y[t]; monotone non-increasing.
        y_vtype = COPT.CONTINUOUS if relax_binaries else COPT.BINARY
        self.y: List[co.Var] = [
            self.m.addVar(lb=0.0, ub=1.0, vtype=y_vtype, name=f"active[{t}]")
            for t in range(self.Tn)
        ]

        for t in range(self.Tn - 1):
            self.m.addConstr(self.y[t] >= self.y[t + 1], name=f"active_noninc[{t}]")

        if self.Tn > 0:
            self.m.addConstr(self.y[0] == 1.0, name="active_start_is_one")

        # Spend convolution (scaled)
        spend_terms: List[List[Tuple[co.Var, int]]] = [[] for _ in range(self.Tn)]
        for (v, s), var in self.x.items():
            vec = spend_by_year[v]
            for k, amtM in enumerate(vec):
                if amtM == 0.0:
                    continue
                t = s + k
                if 0 <= t < self.Tn:
                    spend_terms[t].append((var, iround(amtM, SPEND_SCALE)))

        self.spend_expr: List[co.LinExpr] = []
        for t in range(self.Tn):
            if spend_terms[t]:
                self.spend_expr.append(co.quicksum(coeff * var for (var, coeff) in spend_terms[t]))
            else:
                self.spend_expr.append(co.LinExpr(0.0))

        # If there is spend in year t, then y[t] must be 1.
        for t in range(self.Tn):
            for (var, _coeff) in spend_terms[t]:
                self.m.addConstr(self.y[t] >= var, name=f"active_link[{t}]")

        # Funding drawn each year: fixed to envelope if active.
        self.funding: List[co.Var] = []
        for t in range(self.Tn):
            ub = float(self.funding_target_S[t])
            var = self.m.addVar(lb=0.0, ub=ub, vtype=COPT.CONTINUOUS, name=f"fund[{t}]")
            self.funding.append(var)
            self.m.addConstr(var == self.y[t] * ub, name=f"fund_fixed[{t}]")

        # Dividend variable
        self.dividend: List[co.Var] = []
        for t in range(self.Tn):
            self.dividend.append(
                self.m.addVar(lb=0.0, ub=self.net_ub[t], vtype=COPT.CONTINUOUS, name=f"dividend[{t}]")
            )

        # Net path (non-negative)
        self.net: List[co.Var] = [
            self.m.addVar(lb=0.0, ub=self.net_ub[t], vtype=COPT.CONTINUOUS, name=f"net[{t}]")
            for t in range(self.Tn)
        ]

        if self.Tn > 0:
            self.m.addConstr(self.net[0] == self.funding[0] - self.spend_expr[0] - self.dividend[0], name="net0")
        for t in range(1, self.Tn):
            self.m.addConstr(
                self.net[t] == self.net[t - 1] + self.funding[t] - self.spend_expr[t] - self.dividend[t],
                name=f"net[{t}]",
            )

        # ------------------------------------------------------------------
        # PIECEWISE LINEAR SOFT CAP
        # ------------------------------------------------------------------
        self.excess_tiers: List[List[co.Var]] = [[] for _ in range(self.Tn)]
        self.tier_penalties_expr = co.LinExpr(0.0)

        base_threshold = PIECEWISE_CAP_TIERS[0][0]

        for t in range(self.Tn):
            env_S = float(self.funding_target_S[t])
            base_cap_S = env_S * base_threshold

            tier_vars_t: List[co.Var] = []
            for i, (thresh_start, weight) in enumerate(PIECEWISE_CAP_TIERS):
                is_last = i == len(PIECEWISE_CAP_TIERS) - 1
                if not is_last:
                    thresh_next = PIECEWISE_CAP_TIERS[i + 1][0]
                    width_S = env_S * (thresh_next - thresh_start)
                    v_tier = self.m.addVar(lb=0.0, ub=width_S, vtype=COPT.CONTINUOUS, name=f"exc_t{i}_{t}")
                else:
                    v_tier = self.m.addVar(lb=0.0, vtype=COPT.CONTINUOUS, name=f"exc_t{i}_{t}")

                tier_vars_t.append(v_tier)
                self.tier_penalties_expr.addTerm(v_tier, float(weight))

            self.excess_tiers[t] = tier_vars_t

            if t < self.Tn - 1:
                sum_excess_t = co.quicksum(tier_vars_t)

                if self.use_indicators:
                    self.m.addGenConstrIndicator(
                        self.y[t + 1], True,
                        self.net[t] <= base_cap_S + sum_excess_t,
                        name=f"net_cap_piecewise_ind[{t}]",
                    )
                else:
                    M_t = float(self.net_ub[t])
                    self.m.addConstr(
                        self.net[t] <= base_cap_S + sum_excess_t + M_t * (1.0 - self.y[t + 1]),
                        name=f"net_cap_piecewise[{t}]",
                    )
            else:
                for v in tier_vars_t:
                    self.m.addConstr(v == 0.0, name=f"exc_last_zero[{t}]")

        # ------------------------------------------------------------------
        # Dividend restriction: only allowed at end-of-life (y[t+1]=0)
        # ------------------------------------------------------------------
        for t in range(self.Tn - 1):
            if self.use_indicators:
                self.m.addGenConstrIndicator(
                    self.y[t + 1], True,
                    self.dividend[t] == 0.0,
                    name=f"div_restrict_ind[{t}]",
                )
            else:
                M_t = float(self.net_ub[t])
                self.m.addConstr(
                    self.dividend[t] <= M_t * (1.0 - self.y[t + 1]),
                    name=f"div_restrict[{t}]",
                )

        # ------------------------------------------------------------------
        # Backlog variables: exactly net when still active next year, else 0.
        # ------------------------------------------------------------------
        self.backlog: List[co.Var] = [
            self.m.addVar(lb=0.0, ub=self.net_ub[t], vtype=COPT.CONTINUOUS, name=f"backlog[{t}]")
            for t in range(self.Tn)
        ]
        for t in range(self.Tn):
            if t < self.Tn - 1:
                if self.use_indicators:
                    self.m.addGenConstrIndicator(
                        self.y[t + 1], True,
                        self.backlog[t] == self.net[t],
                        name=f"backlog_eq_net_ind[{t}]",
                    )
                    self.m.addGenConstrIndicator(
                        self.y[t + 1], False,
                        self.backlog[t] == 0.0,
                        name=f"backlog_zero_ind[{t}]",
                    )
                else:
                    M_t = float(self.net_ub[t])
                    self.m.addConstr(self.backlog[t] <= self.net[t], name=f"backlog_le_net[{t}]")
                    self.m.addConstr(self.backlog[t] <= M_t * self.y[t + 1], name=f"backlog_le_My[{t}]")
                    self.m.addConstr(
                        self.backlog[t] >= self.net[t] - M_t * (1.0 - self.y[t + 1]),
                        name=f"backlog_ge_net_M[{t}]",
                    )
            else:
                self.m.addConstr(self.backlog[t] == 0.0, name="backlog_last_zero")

        self.annual_backlog_sum = co.quicksum(self.backlog[t] for t in range(self.Tn))

        # Starts: per-project exactly once; per-year cap
        start_constr = (COPT.EQUAL, 1.0)

        for v in {vv for vv, _ in self.x}:
            cols = [self.x[(vv, s)] for (vv, s) in self.x if vv == v]
            if cols:
                self.m.addConstr(co.quicksum(cols), start_constr[0], start_constr[1], name=f"start_once[{v}]")

        for t in range(self.Tn):
            cols = [self.x[(v, s)] for (v, s) in self.by_year.get(t, [])]
            if cols:
                self.m.addConstr(co.quicksum(cols) <= int(max_starts_per_year), name=f"cap_starts[{t}]")

        if self.x:
            self.m.addConstr(co.quicksum(self.x.values()) >= 1.0, name="at_least_one_project")

        self._obj_cache: Dict[str, co.LinExpr] = {}

    def obj_expr(self, coeff_int_map: Dict[Tuple[str, int], int], cache_key: Optional[str] = None) -> co.LinExpr:
        if cache_key and cache_key in self._obj_cache:
            return self._obj_cache[cache_key]
        terms = []
        for (v, s), w in coeff_int_map.items():
            var = self.x.get((v, s))
            if var is not None and int(w) != 0:
                terms.append(int(w) * var)
        expr = co.quicksum(terms) if terms else co.LinExpr(0.0)
        if cache_key:
            self._obj_cache[cache_key] = expr
        return expr

    def add_floor(self, expr: co.LinExpr, target_unscaled: float, name: str) -> None:
        self.m.addConstr(expr >= int(math.floor(target_unscaled * PV_SCALE)), name=name)

    def add_mip_start_from_selection(self, sel: Dict[Tuple[str, int], int]) -> None:
        if not sel:
            return
        try:
            vars_: List[co.Var] = []
            vals_: List[float] = []
            for (k, on) in sel.items():
                if on and k in self.x:
                    vars_.append(self.x[k])
                    vals_.append(1.0)

            if not vars_:
                return

            if hasattr(self.m, "setMipStart") and hasattr(self.m, "loadMipStart"):
                self.m.setMipStart(vars_, vals_)
                self.m.loadMipStart()
            else:
                if hasattr(self.m, "addMIPStart"):
                    self.m.addMIPStart(vars_, vals_)
        except Exception as e:
            _warn(f"Failed to add MIP start: {e}")


# ─────────────────────────────────────────────────────────────────────
# Solve helpers – v94.21 (single-phase grind; no quick stage)
# ─────────────────────────────────────────────────────────────────────
@dataclass
class SolveResult:
    status: int
    has_inc: bool
    gap: Optional[float]
    seconds: float


def solve_model(
    M: CoptSpendMatchMO,
    stage: str,
    rel_gap: float,
    *,
    time_limit: Optional[float] = None,
) -> SolveResult:
    target_gap = float(rel_gap) if rel_gap and rel_gap > 0 else 0.0001
    HUGE_TIMELIMIT = float(time_limit) if (time_limit is not None and time_limit > 0) else 1e20

    _log(f"[{stage}] Single-phase grind-from-start: target gap={target_gap:.6%}, TimeLimit={HUGE_TIMELIMIT:g}")
    t0 = time.time()

    _apply_params_logged(
        M.m,
        {
            "TimeLimit": float(HUGE_TIMELIMIT),
            "RelGap": float(target_gap),
        },
        header=f"{stage} params",
    )
    _log_effective_parallelism(M.m, where=f"{stage}/solve")

    M.m.solve()

    total_elapsed = time.time() - t0
    final_status = int(M.m.status)
    final_gap = _best_gap(M.m)

    _log(
        f"[{stage}] Final gap={(final_gap if final_gap is not None else float('nan')):.4%}, "
        f"elapsed={total_elapsed:.1f}s"
    )

    return SolveResult(
        status=final_status,
        has_inc=_has_incumbent(M.m),
        gap=final_gap,
        seconds=total_elapsed,
    )


# ─────────────────────────────────────────────────────────────────────
# Selection / PV helpers
# ─────────────────────────────────────────────────────────────────────
def selection_from_values(xmap: Dict[Tuple[str, int], co.Var], val_by_id: Optional[Dict[int, float]]) -> Dict[Tuple[str, int], int]:
    out: Dict[Tuple[str, int], int] = {}
    for key, var in xmap.items():
        if _val(val_by_id, var) > 0.5:
            out[key] = 1
    return out


def pv_from_selection(coeff_map: Dict[Tuple[str, int], float], sel: Dict[Tuple[str, int], int]) -> float:
    return float(sum(coeff_map.get(k, 0.0) for k, on in sel.items() if on))


def extract_solution_values(M: CoptSpendMatchMO) -> Optional[Dict[int, float]]:
    if not _has_incumbent(M.m):
        return None

    val_by_id: Dict[int, float] = {}
    all_vars: List[co.Var] = []
    all_vars.extend(M.x.values())
    all_vars.extend(M.net)
    all_vars.extend(M.funding)
    all_vars.extend(M.y)
    all_vars.extend(M.backlog)
    all_vars.extend(M.dividend)
    for t in range(M.Tn):
        all_vars.extend(M.excess_tiers[t])

    for var in all_vars:
        if var is not None:
            val_by_id[id(var)] = _val(None, var)
    return val_by_id


# ─────────────────────────────────────────────────────────────────────
# Diagnostics / PKL
# ─────────────────────────────────────────────────────────────────────
def assert_and_log_invariants(
    M: CoptSpendMatchMO,
    funding_target_M: np.ndarray,
    val_by_id: Dict[int, float],
    *,
    label: str,
) -> None:
    fy = cal_years(M.Tn)
    net = np.array([_val(val_by_id, M.net[t]) for t in range(M.Tn)], dtype=float)

    if np.any(net < -1e-5):
        viol_t = int(np.argmin(net))
        raise AssertionError(
            f"[{label}] debt detected at year {fy[viol_t]}! "
            f"Net balance went negative: {net[viol_t]/SPEND_SCALE:.3f} M"
        )

    print(f"[diag] {label}:")
    print(
        f"[diag] head net="
        f"{[(fy[i], round(net[i]/SPEND_SCALE, 3)) for i in range(min(5, M.Tn))]} "
        f"tail net="
        f"{[(fy[i], round(net[i]/SPEND_SCALE, 3)) for i in range(max(0, M.Tn-5), M.Tn)]}"
    )
    print(f"[diag] Closing Net Balance (raw model): {net[-1]/SPEND_SCALE:,.2f} M")


def dump_pickle_full(
    M: CoptSpendMatchMO,
    tag: str,
    *,
    projects: Dict[str, dict],
    variants: Dict[str, dict],
    costs_input_df: pd.DataFrame,
    ben_kernel_df: pd.DataFrame,
    kernels_by_dim: Dict[str, Dict[str, List[float]]],
    benefit_rate: float,
    scenario_name: str,
    primary_dim: str,
    Tfine: int,
    funding_target_M: np.ndarray,
    sel_override: Optional[Dict[Tuple[str, int], int]] = None,
    val_override: Optional[Dict[int, float]] = None,
    gap_override: Optional[float] = None,
    extra_diag: Optional[Dict[str, str]] = None,
    status_override: Optional[str] = None,
) -> None:
    ny = Tfine
    fy = cal_years(ny)

    if sel_override is not None:
        sel = sel_override
    elif val_override is not None:
        sel = selection_from_values(M.x, val_override)
    else:
        sel = selection_from_values(M.x, None)

    status_text = status_override or "OK"
    if not sel:
        fn = CACHE / f"{(PKL_PREFIX or '')}{tag}_noSol.pkl"
        payload = {
            "status": "NoSolve",
            "reason": "no selected starts",
            "objective": primary_dim,
            "created_at": _now_stamp(),
            "gap": None,
            "gap_pct": None,
        }
        if extra_diag:
            payload["diagnostic"] = extra_diag
        pkl_save(fn, payload)
        return

    rows = []
    for (v, s) in sorted(sel.keys()):
        rows.append(
            {
                "Project": v,
                "StartFY": START_FY + s,
                "EndFY": START_FY + s + variants[v]["dur"] - 1,
                "Dur": variants[v]["dur"],
                "Scenario": scenario_name,
                "PrimaryDim": primary_dim,
            }
        )
    df_sched = pd.DataFrame(rows).sort_values(["StartFY", "Project"], ignore_index=True)

    df_sp = pd.DataFrame(0.0, index=list(projects.keys()), columns=fy)
    for (v, s) in sel.keys():
        vec = variants[v]["spend"]
        for i, amt in enumerate(vec):
            t = s + i
            if 0 <= t < ny:
                df_sp.loc[v, fy[t]] += float(amt)
    df_sp.loc["Total Spend"] = df_sp.sum()

    total_spend_series = df_sp.loc["Total Spend"]
    last_spend_year = None
    for yr in reversed(fy):
        if abs(float(total_spend_series.get(yr, 0.0))) > 1e-6:
            last_spend_year = yr
            break
    last_spend_idx = -1 if last_spend_year is None else fy.index(last_spend_year)

    val_by_id = val_override or {}

    funding_M_raw: List[float] = []
    for t in range(ny):
        funding_M_raw.append(_val(val_by_id, M.funding[t]) / SPEND_SCALE if val_by_id else float(funding_target_M[t]))

    dividend_M_raw: List[float] = []
    for t in range(ny):
        dividend_M_raw.append(_val(val_by_id, M.dividend[t]) / SPEND_SCALE if val_by_id else 0.0)

    net_M_raw: List[float] = []
    for t in range(ny):
        net_M_raw.append(_val(val_by_id, M.net[t]) / SPEND_SCALE if val_by_id else 0.0)

    funding_M: List[float] = []
    net_M: List[float] = []
    env_M_plot: List[float] = []
    div_M: List[float] = []
    for t in range(ny):
        if last_spend_idx >= 0 and t <= last_spend_idx:
            funding_M.append(funding_M_raw[t])
            net_M.append(net_M_raw[t])
            env_M_plot.append(float(funding_target_M[t]))
            div_M.append(dividend_M_raw[t])
        else:
            funding_M.append(0.0)
            net_M.append(0.0)
            env_M_plot.append(0.0)
            div_M.append(0.0)

    cash_rows = []
    for t in range(ny):
        yr = fy[t]
        spend = float(df_sp.loc["Total Spend", yr])
        env = env_M_plot[t]
        fund = funding_M[t]
        opening_net = 0.0 if t == 0 else net_M[t - 1]
        closing_net = net_M[t]
        div = div_M[t]
        cash_rows.append(
            dict(
                Year=yr,
                Envelope=env,
                Funding=fund,
                OpeningNet=opening_net,
                Spend=spend,
                Dividend=div,
                ClosingNet=closing_net,
                OpeningCash=max(opening_net, 0.0),
                OpeningDebt=max(-opening_net, 0.0),
                ClosingCash=max(closing_net, 0.0),
                ClosingDebt=max(-closing_net, 0.0),
            )
        )
    df_cash = pd.DataFrame(cash_rows)

    dims = list(kernels_by_dim.keys())
    Tstore = ny
    for (v, s) in sel.keys():
        for d in dims:
            ker = kernels_by_dim.get(d, {}).get(v, [])
            if ker:
                Tstore = max(Tstore, s + len(ker))
    fy_store = cal_years(Tstore)

    ben_total_by_dim_store: Dict[str, np.ndarray] = {d: np.zeros(Tstore, float) for d in dims}
    proj_dim_year_store: Dict[Tuple[str, str], np.ndarray] = {
        (p, d): np.zeros(Tstore, float) for d in dims for p in projects.keys()
    }

    for (v, s) in sel.keys():
        for d in dims:
            ker = kernels_by_dim[d].get(v, [])
            for k, f in enumerate(ker):
                t = s + k
                if 0 <= t < Tstore:
                    ben_total_by_dim_store[d][t] += float(f)
                    proj_dim_year_store[(v, d)][t] += float(f)

    df_ben_year = pd.DataFrame({"Year": fy_store, **{d: ben_total_by_dim_store[d] for d in dims}})

    idx = pd.MultiIndex.from_product([list(projects.keys()), dims], names=["Project", "Dimension"])
    df_ben_proj_dim_year = pd.DataFrame(0.0, index=idx, columns=fy_store)
    for (p, d), vec in proj_dim_year_store.items():
        df_ben_proj_dim_year.loc[(p, d), :] = vec

    # PV computed over ny (=40) years, as requested
    disc = np.array([(1.0 + benefit_rate) ** t for t in range(ny)], float)
    pv_by_dim: Dict[str, float] = {d: float(np.sum(ben_total_by_dim_store[d][:ny] / disc)) for d in dims}
    total_pv = pv_by_dim.get("Total", float(np.sum([pv_by_dim[d] for d in dims if d != "Total"])))

    pv_by_proj_dim = pd.DataFrame(0.0, index=idx, columns=["PV"])
    for (p, d) in idx:
        vec_full = df_ben_proj_dim_year.loc[(p, d)].to_numpy(dtype=float)
        pv_by_proj_dim.loc[(p, d), "PV"] = float(np.sum(vec_full[:ny] / disc))

    try:
        val_for_diag = val_override or {id(var): float(var.X) for var in M.net}
        assert_and_log_invariants(M, funding_target_M, val_for_diag, label=tag)
        net_list = [_val(val_for_diag, M.net[t]) / SPEND_SCALE for t in range(M.Tn)]
        diag_caps: Dict[str, Any] = {
            "closing_net_M_raw": float(net_list[-1]),
            "max_net_M_raw": float(max(net_list) if net_list else 0.0),
        }
    except AssertionError as e:
        diag_caps = {"assertion_error": str(e)}

    gap_now = float(gap_override) if gap_override is not None else _best_gap(M.m)
    gap_pct = gap_now * 100.0 if gap_now is not None else None

    out: Dict[str, Any] = dict(
        status=status_text,
        objective=primary_dim,
        created_at=_now_stamp(),
        scenario=scenario_name,
        primary_dim=primary_dim,
        schedule=df_sched,
        spend=df_sp,
        cash_flow=df_cash,
        envelope=pd.DataFrame({"Year": fy, "Envelope": env_M_plot}),
        benefits_by_year=df_ben_year,
        benefits_by_project_dimension_by_year=df_ben_proj_dim_year,
        pv_by_dimension=pv_by_dim,
        pv_total=total_pv,
        gap=gap_now,
        gap_pct=gap_pct,
        best={"pv": total_pv, "gap": gap_now, "gap_pct": gap_pct},
        pv_by_project_and_dimension=pv_by_proj_dim,
        calendar=dict(start_fy=START_FY, years=ny),
        meta=dict(
            full_envelope_M=float(funding_target_M[0]) if len(funding_target_M) > 0 else 0.0,
            taper_years=0,
            backlog_weight=BACKLOG_WEIGHT,
            pv_weight=PV_WEIGHT,
            net_cap_tiers=PIECEWISE_CAP_TIERS,
            run_id=RUN_ID,
            threads=SOLVER_THREADS_DEFAULT,
            tasks=SOLVER_TASKS_DEFAULT,
            use_indicators=USE_INDICATOR_CONSTRAINTS,
        ),
    )
    if extra_diag:
        out.setdefault("diagnostic", {}).update(extra_diag)
    if diag_caps:
        out.setdefault("diagnostic", {}).update(diag_caps)

    fn = CACHE / f"{(PKL_PREFIX or '')}{tag}.pkl"
    if str(fn).endswith("}.pkl"):
        fn = Path(str(fn)[:-5] + ".pkl")
    pkl_save(fn, out)


# ─────────────────────────────────────────────────────────────────────
# Cross‑buffer registry
# ─────────────────────────────────────────────────────────────────────
_BEST_PV_BY_ENV: Dict[Tuple[str, str, float], float] = {}
_BEST_SEL_BY_ENV: Dict[Tuple[str, str, float], Dict[Tuple[str, int], int]] = {}

# ─────────────────────────────────────────────────────────────────────
# Objective construction
# ─────────────────────────────────────────────────────────────────────
def _set_weighted_objective(M: CoptSpendMatchMO, expr_primary_pv: co.LinExpr, *, tag: str) -> None:
    obj_backlog = BACKLOG_WEIGHT * M.annual_backlog_sum
    obj_excess = M.tier_penalties_expr
    obj_pv = expr_primary_pv * PV_WEIGHT

    combined_obj = obj_backlog + obj_excess - obj_pv
    M.m.setObjective(combined_obj, COPT.MINIMIZE)

    _log(f"[{tag}] Objective: MIN( {BACKLOG_WEIGHT:.1f} * Backlog + PiecewiseExcessPenalty - {PV_WEIGHT:.1e} * PV )")


@dataclass
class Incumbent:
    sel: Dict[Tuple[str, int], int]
    pv: float
    max_net_balance: float
    annual_backlog_sum: float
    val_by_id: Dict[int, float]
    model: CoptSpendMatchMO
    gap: Optional[float]


def build_total_model(
    variants: Dict[str, dict],
    allowed: Dict[str, List[int]],
    funding_target_S: np.ndarray,
    Tfine: int,
    taper_start_idx: int,
    coeff_total_fine_int: Dict[Tuple[str, int], int],
    total_floor_target: Optional[float],
    is_whitelist_model: bool,
    *,
    relax_binaries: bool = False,
) -> CoptSpendMatchMO:
    M = CoptSpendMatchMO(
        variants,
        allowed,
        funding_target_S,
        Tfine,
        taper_start_idx,
        spend_by_year={v: variants[v]["spend"] for v in variants},
        max_starts_per_year=MAX_STARTS_PER_FY,
        is_whitelist_model=is_whitelist_model,
        name="MO_SPEND_MATCH",
        relax_binaries=relax_binaries,
        use_indicators=USE_INDICATOR_CONSTRAINTS,
    )

    expr_tot_pv = M.obj_expr(coeff_total_fine_int, cache_key="TOT_PV")

    if total_floor_target is not None:
        floor_target = total_floor_target - max(MONO_ABS_EPS, MONO_REL_EPS * abs(total_floor_target))
        M.add_floor(expr_tot_pv, floor_target, name="mono_total_floor")

    _set_weighted_objective(M, expr_tot_pv, tag="TOTAL")
    return M


def eval_incumbent(M: CoptSpendMatchMO, val_by_id: Dict[int, float], coeff_total_fine: Dict[Tuple[str, int], float]) -> Incumbent:
    sel = selection_from_values(M.x, val_by_id)
    pv = pv_from_selection(coeff_total_fine, sel)

    net_vals_M = [_val(val_by_id, M.net[t]) / SPEND_SCALE for t in range(M.Tn)]
    max_net_balance = max(net_vals_M) if net_vals_M else 0.0

    backlog_vals_M = [_val(val_by_id, M.backlog[t]) / SPEND_SCALE for t in range(M.Tn)]
    annual_backlog_sum = float(sum(backlog_vals_M))

    gap = _best_gap(M.m)
    return Incumbent(sel=sel, pv=pv, max_net_balance=max_net_balance, annual_backlog_sum=annual_backlog_sum, val_by_id=val_by_id, model=M, gap=gap)


def orchestrate_total(
    variants: Dict[str, dict],
    allowed_fine: Dict[str, List[int]],
    funding_target_S: np.ndarray,
    Tfine: int,
    taper_start_idx: int,
    coeff_total_fine: Dict[Tuple[str, int], float],
    coeff_total_fine_int: Dict[Tuple[str, int], int],
    total_floor_target: Optional[float],
    is_whitelist_model: bool,
    *,
    mip_start_candidates: List[Dict[Tuple[str, int], int]],
    rel_gap_target: float,
) -> Tuple[Optional[Incumbent], Optional[float]]:
    _log(
        f"[orchestrate] Building model with FIXED FUNDING + PIECEWISE SOFT CAP. "
        f"Single-phase solve; target gap={rel_gap_target:.6%}"
    )

    M = build_total_model(
        variants,
        allowed_fine,
        funding_target_S,
        Tfine,
        taper_start_idx,
        coeff_total_fine_int,
        total_floor_target,
        is_whitelist_model,
    )

    for sel in mip_start_candidates:
        M.add_mip_start_from_selection(sel)

    res = solve_model(
        M,
        stage="MAIN_SOLVE",
        rel_gap=rel_gap_target,
    )

    if not res.has_inc:
        _log("[orchestrate] Solve finished with no incumbent solution.")
        return None, res.gap

    valmap = extract_solution_values(M)
    if not valmap:
        _log("[orchestrate] Solve failed to extract variable values.")
        return None, res.gap

    best_inc = eval_incumbent(M, valmap, coeff_total_fine)
    _log(
        "[orchestrate] Solve complete. "
        f"BacklogSum: {best_inc.annual_backlog_sum:,.1f} M, "
        f"MaxNetBalance: {best_inc.max_net_balance:,.1f} M, "
        f"PV: {best_inc.pv:,.1f}, "
        f"Gap: {best_inc.gap or float('nan'):.4%}"
    )
    return best_inc, best_inc.gap


# ─────────────────────────────────────────────────────────────────────
# DIMENSION helpers
# ─────────────────────────────────────────────────────────────────────
TOTAL_PV_GUARD_PCT = 85.0

_DIM_SHORT: Dict[str, str] = {
    "Total": "TOT",
    "Healthy and safe people": "HSP",
    "Inclusive Access": "INC",
    "Economic Prosperity": "ECO",
}


def dim_short(dim: str) -> str:
    if dim in _DIM_SHORT:
        return _DIM_SHORT[dim]
    toks = re.findall(r"[A-Za-z0-9]+", dim or "")
    return ("".join(t[:3] for t in toks)[:8] or "DIM").upper()


def weights_for_dimension(
    primary_dim: str,
    variants: Dict[str, dict],
    kernels_by_dim: Dict[str, Dict[str, List[float]]],
    r: float,
) -> Dict[str, float]:
    alpha = 1.0
    beta = 0.5
    wmin = 1.0
    wmax = 2.0
    eps = 1e-9
    dims = list(kernels_by_dim.keys())
    if primary_dim not in kernels_by_dim:
        return {v: 1.0 for v in variants}

    def pv_start0(ker: List[float], r_: float) -> float:
        pv = 0.0
        denom = 1.0
        g = 1.0 + r_
        for f in ker:
            pv += float(f) / denom
            denom *= g
        return pv

    pv0_primary = {v: pv_start0(kernels_by_dim[primary_dim].get(v, []), r) for v in variants}
    arr = np.array(list(pv0_primary.values()))
    order = np.argsort(arr)
    ranks = np.empty_like(arr, dtype=float)
    ranks[order] = np.arange(1, len(arr) + 1, dtype=float)
    pct = {k: float(max(ranks[i] / float(len(arr)), eps)) for i, k in enumerate(pv0_primary.keys())}

    others = [d for d in dims if d != primary_dim and d.lower() != "total"]
    pv0_other_mean: Dict[str, float] = {}
    for v in variants:
        vals = [pv_start0(kernels_by_dim.get(d, {}).get(v, []), r) for d in others if kernels_by_dim.get(d, {}).get(v)]
        pv0_other_mean[v] = float(np.mean(vals)) if vals else 0.0

    ratio: Dict[str, float] = {}
    for v in variants:
        denom = max(pv0_other_mean[v], eps)
        ratio[v] = pv0_primary[v] / denom if denom > 0 else (wmax if pv0_primary[v] > 0 else 1.0)

    w: Dict[str, float] = {}
    for v in variants:
        wv = (max(ratio[v], eps)) ** alpha * (max(pct[v], eps)) ** beta
        w[v] = float(wv)
    m = float(np.mean(list(w.values()))) if w else 1.0
    if m > eps:
        w = {k: v / m for k, v in w.items()}

    return {k: float(min(max(v, wmin), wmax)) for k, v in w.items()}


def coeff_map_for_dim_fine_weighted(
    dim: str,
    variants: Dict[str, dict],
    kernels_by_dim: Dict[str, Dict[str, List[float]]],
    allowed_fine: Dict[str, List[int]],
    Tfine: int,
    disc_vec: np.ndarray,
) -> Tuple[Dict[Tuple[str, int], float], Dict[Tuple[str, int], int]]:
    w = weights_for_dimension(dim, variants, kernels_by_dim, BENEFIT_DISCOUNT_RATE)
    base = coeff_map_for_dim_fine(variants, kernels_by_dim.get(dim, {}), allowed_fine, Tfine, disc_vec)
    if not base:
        return {}, {}
    weighted = {(v, s): base.get((v, s), 0.0) * float(w.get(v, 1.0)) for (v, s) in base}
    return weighted, coeff_int(weighted)


def run_dimensions_for_env(
    ct: str,
    sc_key: str,
    sc_sheet: str,
    sur_key: str,
    plus_M: float,
    *,
    projects: Dict[str, dict],
    variants: Dict[str, dict],
    costs_input_df: pd.DataFrame,
    ben_kernel_df: pd.DataFrame,
    kernels_by_dim: Dict[str, Dict[str, List[float]]],
    Tfine: int,
    taper_start_idx: int,
    funding_target_M: np.ndarray,
    funding_target_S: np.ndarray,
    allowed_fine: Dict[str, List[int]],
    disc_vec: np.ndarray,
    coeff_total_fine_int: Dict[Tuple[str, int], int],
    total_best_sel: Dict[Tuple[str, int], int],
    total_best_pv: float,
    is_whitelist_model: bool,
    tot_model_for_fallback: Optional[CoptSpendMatchMO],
    tot_valmap_for_fallback: Optional[Dict[int, float]],
    prev_dim_floors: Optional[Dict[str, float]] = None,
    dims_filter: Optional[Iterable[str]] = None,
) -> Dict[str, float]:
    prev_dim_floors = dict(prev_dim_floors or {})
    dims_all = [d for d in kernels_by_dim.keys() if d.lower() != "total"]
    dims = [d for d in dims_all if d in set(dims_filter)] if dims_filter is not None else dims_all
    if not dims:
        _log(" [DIM] No weighted-dimension runs requested.")
        return prev_dim_floors

    total_floor = float(total_best_pv) * (TOTAL_PV_GUARD_PCT / 100.0)

    for dim in dims:
        tag_dim = (
            f"{ct.replace(' ','').replace('-', '')}_{sc_key}_{sur_key}_pm{int(plus_M)}_"
            f"{dim_short(dim)}"
        )
        _log(f" [DIM] {dim} (weighted) with Total-guard ≥ {total_floor:,.6f}")

        coeff_dim_w_f, coeff_dim_w_int = coeff_map_for_dim_fine_weighted(
            dim, variants, kernels_by_dim, allowed_fine, Tfine, disc_vec
        )

        if not coeff_dim_w_int:
            if tot_model_for_fallback is not None:
                dump_pickle_full(
                    tot_model_for_fallback,
                    tag_dim,
                    projects=projects,
                    variants=variants,
                    costs_input_df=costs_input_df,
                    ben_kernel_df=ben_kernel_df,
                    kernels_by_dim=kernels_by_dim,
                    benefit_rate=BENEFIT_DISCOUNT_RATE,
                    scenario_name=sc_key,
                    primary_dim=f"{dim} [WEIGHTED]",
                    Tfine=Tfine,
                    funding_target_M=funding_target_M,
                    sel_override=total_best_sel,
                    val_override=tot_valmap_for_fallback,
                    gap_override=_best_gap(tot_model_for_fallback.m),
                    status_override="OK_SUBOPT_NOCOEFF",
                    extra_diag={"note": "dimension has zero coefficients; reported Total solution"},
                )
            prev_dim_floors[dim] = 0.0
            continue

        Md = CoptSpendMatchMO(
            variants,
            allowed_fine,
            funding_target_S,
            Tfine,
            taper_start_idx,
            spend_by_year={v: variants[v]["spend"] for v in variants},
            max_starts_per_year=MAX_STARTS_PER_FY,
            is_whitelist_model=is_whitelist_model,
            name=f"MO_DIM_{dim_short(dim)}",
            use_indicators=USE_INDICATOR_CONSTRAINTS,
        )

        Md.add_mip_start_from_selection(total_best_sel)

        expr_dim_pv = Md.obj_expr(coeff_dim_w_int, cache_key=f"DIM_{dim_short(dim)}")
        expr_tot_guard = Md.obj_expr(coeff_total_fine_int, cache_key="TOT_GUARD")

        Md.add_floor(expr_tot_guard, total_floor - max(MONO_ABS_EPS, MONO_REL_EPS * abs(total_floor)), name="dim_total_floor")

        _set_weighted_objective(Md, expr_dim_pv, tag=f"DIM_{dim_short(dim)}")

        rel_gap = EFFORT[OPTIMISATION_PROFILE]["REL_GAP"]
        res = solve_model(Md, stage=f"DIM[{dim_short(dim)}]", rel_gap=rel_gap)

        if _has_incumbent(Md.m):
            valmap = extract_solution_values(Md)
            if valmap:
                assert_and_log_invariants(Md, funding_target_M, valmap, label=tag_dim)
                dump_pickle_full(
                    Md,
                    tag_dim,
                    projects=projects,
                    variants=variants,
                    costs_input_df=costs_input_df,
                    ben_kernel_df=ben_kernel_df,
                    kernels_by_dim=kernels_by_dim,
                    benefit_rate=BENEFIT_DISCOUNT_RATE,
                    scenario_name=sc_key,
                    primary_dim=f"{dim} [WEIGHTED]",
                    Tfine=Tfine,
                    funding_target_M=funding_target_M,
                    sel_override=selection_from_values(Md.x, valmap),
                    val_override=valmap,
                    gap_override=res.gap,
                )
        else:
            _log(f"[DIM] {dim} solve failed to find incumbent.")
            if tot_model_for_fallback is not None:
                dump_pickle_full(
                    tot_model_for_fallback,
                    tag_dim,
                    projects=projects,
                    variants=variants,
                    costs_input_df=costs_input_df,
                    ben_kernel_df=ben_kernel_df,
                    kernels_by_dim=kernels_by_dim,
                    benefit_rate=BENEFIT_DISCOUNT_RATE,
                    scenario_name=sc_key,
                    primary_dim=f"{dim} [WEIGHTED]",
                    Tfine=Tfine,
                    funding_target_M=funding_target_M,
                    sel_override=total_best_sel,
                    val_override=tot_valmap_for_fallback,
                    gap_override=_best_gap(tot_model_for_fallback.m),
                    status_override="OK_SUBOPT_NOSOLVE",
                    extra_diag={"note": "dimension solver yielded no incumbent; reported Total solution"},
                )

    return prev_dim_floors


# ─────────────────────────────────────────────────────────────────────
# Run combo
# ─────────────────────────────────────────────────────────────────────
def get_funding_envelope(Tfine: int, base_M: float, taper_start_from_end: int) -> Tuple[np.ndarray, int]:
    funding_M = np.full(Tfine, float(base_M), dtype=float)
    taper_start_idx = Tfine
    return funding_M, taper_start_idx


def run_combo(
    ct: str,
    sc_key: str,
    sc_sheet: str,
    sur_key: str,
    baseline_surplus_M: float,
    plus_M: float,
    prev_dim_floors: Optional[Dict[str, float]] = None,
    prev_best_sel: Optional[Dict[Tuple[str, int], int]] = None,
):
    projects_all, variants_all, costs_input_df = load_costs(ct)

    variants, forced_exact, is_whitelist_model = apply_forced_rules(variants_all, FORCED_START)
    projects = {p: projects_all[p] for p in variants.keys() if p in projects_all}

    min_start_norm = {norm(k): v for k, v in (MIN_START_YEAR or {}).items()}
    min_start_exact: Dict[str, int] = {}
    for v in variants:
        n = norm(v)
        if n in min_start_norm:
            min_start_exact[v] = min_start_norm[n]

    benef_df, ben_kernel_df = load_benefits(sc_sheet)
    _dims_order, kernels_by_dim = map_benefit_kernels(benef_df, variants)
    total_key = "Total"

    include_norm = {norm(k): bool(v) for k, v in (DIMENSION_INCLUSIONS or {}).items()}
    dims_to_run = [d for d in kernels_by_dim.keys() if d.lower() != "total" and include_norm.get(norm(d), False)]

    full_envelope_M = float(baseline_surplus_M) + float(plus_M)
    Tfine = TFIXED  # 40-year horizon

    tot_cost = float(sum(sum(meta["spend"]) for meta in variants.values()))
    _log(f" Fixed Horizon: TotCost={tot_cost:,.0f}M, Env={full_envelope_M:,.0f}M/yr -> Years={Tfine}")
    _log(f" PV window: {START_FY}→{START_FY + Tfine - 1} (Tfine={Tfine})")

    funding_target_M, taper_start_idx = get_funding_envelope(Tfine, full_envelope_M, taper_start_from_end=TAPER_YEARS)
    funding_target_S = np.array([iround(m, SPEND_SCALE) for m in funding_target_M], dtype=float)
    _log(f" Envelope capacity: up to {full_envelope_M:,.0f} M per year (chosen as needed).")

    tag_tot = f"{ct.replace(' ','').replace('-', '')}_{sc_key}_{sur_key}_pm{int(plus_M)}_{_DIM_SHORT['Total']}"

    total_funding_M = float(np.sum(funding_target_M))
    if is_whitelist_model and (tot_cost > total_funding_M + 1e-9):
        msg = (
            f"[WHITELIST] Total project spend {tot_cost:,.1f} M exceeds *total* "
            f"funding capacity {total_funding_M:,.1f} M. Scenario infeasible."
        )
        print(" ×", msg)
        fn = CACHE / f"{(PKL_PREFIX or '')}{tag_tot}_noSol.pkl"
        pkl_save(
            fn,
            {
                "status": "NoSolve",
                "reason": "mass-balance infeasible (whitelist)",
                "detail": msg,
                "objective": "Total",
                "created_at": _now_stamp(),
                "gap": None,
                "gap_pct": None,
            },
        )
        return None, prev_dim_floors, prev_best_sel

    allowed_fine = allowed_starts_fine(variants, forced_exact, min_start_exact, Tfine)
    missing = [p for p in variants if not allowed_fine.get(p)]
    if missing:
        fn = CACHE / f"{(PKL_PREFIX or '')}{tag_tot}_noSol.pkl"
        pkl_save(
            fn,
            {
                "status": "NoSolve",
                "reason": f"no allowed starts for {missing[:5]}",
                "diagnostic": {"phase": "screen", "Tfine": Tfine},
                "objective": "Total",
                "created_at": _now_stamp(),
                "gap": None,
                "gap_pct": None,
            },
        )
        return None, prev_dim_floors, prev_best_sel

    disc_vec = np.array([(1.0 + BENEFIT_DISCOUNT_RATE) ** t for t in range(Tfine)], dtype=float)
    coeff_total_fine = coeff_map_for_dim_fine(variants, kernels_by_dim[total_key], allowed_fine, Tfine, disc_vec)
    coeff_total_fine_int = coeff_int(coeff_total_fine)

    mip_starts = build_diversified_mip_starts(
        variants,
        allowed_fine,
        Tfine,
        funding_target_M,
        MAX_STARTS_PER_FY,
        forced_exact,
        coeff_total_fine,
        reuse_sel=prev_best_sel,
        n_starts=MULTI_START_COUNT,
    )

    prev_env = max([e for (ctk, sck, e) in _BEST_PV_BY_ENV.keys() if ctk == ct and sck == sc_key and e < full_envelope_M], default=None)
    total_floor_target = None
    if ENFORCE_MONOTONE_PV_ACROSS_BUFFERS and prev_env is not None:
        total_floor_target = _BEST_PV_BY_ENV[(ct, sc_key, prev_env)]

    prof_effort = EFFORT[OPTIMISATION_PROFILE]
    best_inc, final_gap = orchestrate_total(
        variants,
        allowed_fine,
        funding_target_S,
        Tfine,
        taper_start_idx,
        coeff_total_fine,
        coeff_total_fine_int,
        total_floor_target,
        is_whitelist_model,
        mip_start_candidates=mip_starts,
        rel_gap_target=prof_effort["REL_GAP"],
    )

    if best_inc is None or best_inc.model is None:
        fn = CACHE / f"{(PKL_PREFIX or '')}{tag_tot}_noSol.pkl"
        pkl_save(
            fn,
            {
                "status": "NoSolve",
                "reason": "no incumbent with piecewise cap",
                "objective": "Total",
                "created_at": _now_stamp(),
                "diagnostic": {"note": "No feasible solution found."},
                "gap": None,
                "gap_pct": None,
            },
        )
        return None, prev_dim_floors, prev_best_sel

    extra_diag = {
        "orchestrated_gap": f"{(final_gap if final_gap is not None else float('nan')):.4%}",
        "net_cap_tiers": str(PIECEWISE_CAP_TIERS),
    }

    assert_and_log_invariants(best_inc.model, funding_target_M, best_inc.val_by_id, label=tag_tot)

    dump_pickle_full(
        best_inc.model,
        tag_tot,
        projects=projects,
        variants=variants,
        costs_input_df=costs_input_df,
        ben_kernel_df=ben_kernel_df,
        kernels_by_dim=kernels_by_dim,
        benefit_rate=BENEFIT_DISCOUNT_RATE,
        scenario_name=sc_key,
        primary_dim="Total",
        Tfine=Tfine,
        funding_target_M=funding_target_M,
        sel_override=best_inc.sel,
        val_override=best_inc.val_by_id,
        gap_override=best_inc.gap,
        extra_diag=extra_diag,
    )

    best_sel_for_next = best_inc.sel
    valmap_for_fallback = best_inc.val_by_id

    _BEST_PV_BY_ENV[(ct, sc_key, full_envelope_M)] = pv_from_selection(coeff_total_fine, best_sel_for_next)
    if best_sel_for_next:
        _BEST_SEL_BY_ENV[(ct, sc_key, full_envelope_M)] = best_sel_for_next

    disc_vec = np.array([(1.0 + BENEFIT_DISCOUNT_RATE) ** t for t in range(Tfine)], dtype=float)
    prev_dims = run_dimensions_for_env(
        ct,
        sc_key,
        sc_sheet,
        sur_key,
        plus_M,
        projects=projects,
        variants=variants,
        costs_input_df=costs_input_df,
        ben_kernel_df=ben_kernel_df,
        kernels_by_dim=kernels_by_dim,
        Tfine=Tfine,
        taper_start_idx=taper_start_idx,
        funding_target_M=funding_target_M,
        funding_target_S=funding_target_S,
        allowed_fine=allowed_fine,
        disc_vec=disc_vec,
        coeff_total_fine_int=coeff_total_fine_int,
        total_best_sel=best_sel_for_next,
        total_best_pv=pv_from_selection(coeff_total_fine, best_sel_for_next),
        is_whitelist_model=is_whitelist_model,
        tot_model_for_fallback=best_inc.model,
        tot_valmap_for_fallback=valmap_for_fallback,
        prev_dim_floors=prev_dim_floors,
        dims_filter=dims_to_run,
    )

    return (pv_from_selection(coeff_total_fine, best_sel_for_next), prev_dims, best_sel_for_next)


# ─────────────────────────────────────────────────────────────────────
# Driver
# ─────────────────────────────────────────────────────────────────────
def main() -> None:
    random.seed(SOLVER_SEED_DEFAULT)
    np.random.seed(SOLVER_SEED_DEFAULT)

    global _BEST_PV_BY_ENV, _BEST_SEL_BY_ENV
    _BEST_PV_BY_ENV.clear()
    _BEST_SEL_BY_ENV.clear()

    print(
        f"CFG: START_FY={START_FY} FINAL_YEAR={FINAL_YEAR} "
        f"PV_WINDOW={TFIXED}y MAX_STARTS/FY={MAX_STARTS_PER_FY}"
    )
    eff = EFFORT[OPTIMISATION_PROFILE]
    print(f"Effort: {OPTIMISATION_PROFILE} → {{'MO': {eff['MO']}, 'REL_GAP': {eff['REL_GAP']}}}")
    print("NEW MODEL (v94.21):")
    print("  - FIXED FUNDING: funding[t] == Envelope when active.")
    print("  - DIVIDEND: Restricted to end-of-life (y[t+1]=0).")
    print("  - PIECEWISE SOFT CAP: tiers 12%/15%/20%.")
    print("  - ALL PROJECTS: start exactly once.")
    print("  - FIXED PV HORIZON: Tfine = TFIXED.")
    print("  - SOLVE STRATEGY: single-phase grind from start (no quick stage).")
    print(f"  - PARALLEL: Threads={SOLVER_THREADS_DEFAULT}, MipTasks={SOLVER_TASKS_DEFAULT}")
    print(f"  - INDICATORS: {USE_INDICATOR_CONSTRAINTS}\n")

    print(f"DATA_FILE: {DATA_FILE}")
    print(f"CACHE: {CACHE}\n")

    if FORCED_START:
        print(f"FORCED_START: {len(FORCED_START)} rules are active.")
    if MIN_START_YEAR:
        print(f"MIN_START_YEAR: {len(MIN_START_YEAR)} constraint rules are active.")

    dims_incl_str = ", ".join([k for k, v in DIMENSION_INCLUSIONS.items() if k.lower() != "total" and v])
    print(f"Dimension inclusions (non-Total): {dims_incl_str if dims_incl_str else 'None'}")
    print(f"RUN_ID: {RUN_ID}\n")

    for ct in COST_TYPES_RUN:
        print(f"\n=== COST: {ct} ===")
        for sc_key, sc_sheet in BENEFIT_SCENARIOS.items():
            print(f">> Scenario {sc_key} (sheet {sc_sheet})")
            prev_sel: Optional[Dict[Tuple[str, int], int]] = None
            prev_dims: Dict[str, float] = {}
            envs_sorted = sorted(SURPLUS_OPTIONS_M.items(), key=lambda kv: kv[1])

            for sur_key, baseM in envs_sorted:
                print(f" -- Base Funding {sur_key} = {baseM:.0f} M p.a. --")
                for plus in sorted(PLUSMINUS_LEVELS_M):
                    print(f" Running... FULL={baseM+plus:,.0f} M")
                    out = run_combo(
                        ct,
                        sc_key,
                        sc_sheet,
                        sur_key,
                        baseM,
                        plus,
                        prev_dim_floors=prev_dims,
                        prev_best_sel=prev_sel,
                    )
                    if out is not None:
                        tot_pv, prev_dims, prev_sel = out

    print("\n✓ Done. Pickles are under:", CACHE)


if __name__ == "__main__":
    t0 = time.time()
    try:
        main()
    finally:
        print(f"\nRun-time: {time.time()-t0:.1f}s → {CACHE}")


CFG: START_FY=2026 FINAL_YEAR=2065 PV_WINDOW=40y MAX_STARTS/FY=100
Effort: ultra → {'MO': 900.0, 'REL_GAP': 0.0001}
NEW MODEL (v94.21):
  - FIXED FUNDING: funding[t] == Envelope when active.
  - DIVIDEND: Restricted to end-of-life (y[t+1]=0).
  - PIECEWISE SOFT CAP: tiers 12%/15%/20%.
  - ALL PROJECTS: start exactly once.
  - FIXED PV HORIZON: Tfine = TFIXED.
  - SOLVE STRATEGY: single-phase grind from start (no quick stage).
  - PARALLEL: Threads=20, MipTasks=20
  - INDICATORS: True

DATA_FILE: C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation\dummy_cost_benefits.xlsx
CACHE: C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation\scenario_cache_dummy_benefit_mo

Dimension inclusions (non-Total): Economic Prosperity, Healthy and safe people, Inclusive Access
RUN_ID: 7ffa1179


=== COST: P50 - Real ===
>> Scenario DUMMY (sheet Benefits)
 -- Base Funding s500 = 500 M p.a. --
 Running... FULL=500 M
 [rules] Mode=BLACKLIST (auto); kept=50/50; forced=0
 Fixed

Produce pkl file from solution csv schedule

In [1]:
from __future__ import annotations

import re
import time
import pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
ROOT = Path(r"C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation")

# NEW dummy workbook (Costs + Benefits)
DATA_FILE = ROOT / "dummy_cost_benefits.xlsx"
COSTS_SHEET = "Costs"
BENEFITS_SHEET = "Benefits"

# EXACT SAME CSV containing the solution schedule
MODEL_TOT_FILE = ROOT / "modelTOT.csv"

# Optional filters (only applied if matching columns exist in modelTOT.csv)
MODEL_RUN = "s1000"                # model run identifier
MODEL_BUFFER = 0                   # buffer / plus-minus level (pm0)
MODEL_DIMENSION = "Total"          # Total dimension only
MODEL_COST_SCENARIO = "P50 - Real" # cost type

# Output cache folder and target PKL name
CACHE = ROOT / "scenario_cache_benefit_mo"
CACHE.mkdir(exist_ok=True)

# Default output filename (change if you want)
PKL_NAME = "HK_dummy_P50Real_DUMMY_s1000_pm0_TOT.pkl"

# Horizon / calendar (UPDATED: 40-year horizon)
START_FY = 2026
EVALUATION_YEARS = 40
FINAL_YEAR = START_FY + EVALUATION_YEARS - 1
TFIXED = int(FINAL_YEAR - START_FY + 1)
YEARS = TFIXED

# Economics
BENEFIT_DISCOUNT_RATE = 0.02  # 2%

# Funding envelope: s1000 pm0 → 1000 M per year
FULL_ENVELOPE_M = 1000.0

# Objective metadata (for completeness)
BACKLOG_WEIGHT = 1.0
PV_WEIGHT = 1e-4

# Piecewise soft cap tiers (meta only)
PIECEWISE_CAP_TIERS: List[Tuple[float, float]] = [
    (0.12, 1000.0),
    (0.15, 4000.0),
    (0.20, 12000.0),
]

RUN_ID = "GPS27"


# ─────────────────────────────────────────────────────────────
# BASIC HELPERS
# ─────────────────────────────────────────────────────────────
def _now_stamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S", time.localtime())


def clean(s: str) -> str:
    """Strip whitespace and non-breaking spaces."""
    return re.sub(r"\s+", " ", str(s or "").replace("\xa0", " ")).strip()


def norm_key(name: str) -> str:
    """
    Project key normalisation:
    - clean the string
    - drop spaces and underscores
    - lowercase
    So 'project 1' and 'project_1' both become 'project1'.
    """
    return re.sub(r"[\s_]+", "", clean(name)).lower()


def cal_years(ny: int) -> List[int]:
    return [START_FY + i for i in range(ny)]


def pkl_save(path: Path, payload: Dict[str, Any]) -> None:
    """Save payload to pickle and print a short summary."""
    with path.open("wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    size = path.stat().st_size if path.exists() else 0
    objective = payload.get("objective", payload.get("primary_dim", "?"))
    best = payload.get("best", {}) or {}
    pv = best.get("pv", payload.get("pv_total", "?"))
    gap_frac = best.get("gap", payload.get("gap", None))
    gap_pct = best.get("gap_pct", payload.get("gap_pct", None))
    if isinstance(gap_pct, (int, float)):
        gap_str = f"{gap_pct:.4f}%"
    elif isinstance(gap_frac, (int, float)):
        gap_str = f"{gap_frac * 100.0:.4f}%"
    else:
        gap_str = "n/a"

    print(
        f"Saved: {path} ({size} bytes) "
        f"objective={objective} pv={pv} gap={gap_str}"
    )


# ─────────────────────────────────────────────────────────────
# ROBUST NUMBER PARSING (for dummy xlsx)
# ─────────────────────────────────────────────────────────────
_DASH_LITERALS = {"-", "–", "—", " - ", " -- ", "--", "—-", "–-", "n/a", "na", "null", "none", ""}


def to_float(x: Any) -> float:
    """
    Robust conversion for cells like:
      - "106,839,620.21"
      - " - "
      - "(123.45)"
    """
    if x is None:
        return 0.0
    if isinstance(x, (int, float, np.integer, np.floating)):
        try:
            if float(x) != float(x):  # NaN
                return 0.0
        except Exception:
            return 0.0
        return float(x)

    s = clean(str(x))
    if norm_key(s) in {norm_key(v) for v in _DASH_LITERALS}:
        return 0.0

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1].strip()

    s = s.replace(",", "")
    s = re.sub(r"[^0-9eE\.\-\+]", "", s).strip()

    if s in {"", "-", "+", ".", "-.", "+."}:
        return 0.0

    try:
        v = float(s)
    except Exception:
        return 0.0
    return -v if neg else v


def _col_lookup(df: pd.DataFrame, want: str) -> Optional[str]:
    w = norm_key(want)
    for c in df.columns:
        if norm_key(str(c)) == w:
            return str(c)
    return None


# ─────────────────────────────────────────────────────────────
# COSTS (UPDATED FOR dummy_cost_benefits.xlsx)
# ─────────────────────────────────────────────────────────────
def load_costs(cost_type: str):
    """
    Dummy Costs sheet supported format:
      Project | Cost type | ... | Duration | 2026 | 2027 | ... (maybe not all years present)

    - Robust to strings with commas and dash placeholders.
    - Aggregates multiple rows per project (sums year streams).
    - Uses 'Duration' if present (mode if multiple), else falls back to non-zero span.
    - Returns spend in millions (M).
    """
    df = pd.read_excel(DATA_FILE, sheet_name=COSTS_SHEET, engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    proj_col = _col_lookup(df, "Project")
    if not proj_col:
        raise RuntimeError(f"Costs sheet '{COSTS_SHEET}' needs a 'Project' column.")

    cost_type_col = _col_lookup(df, "Cost type")
    if not cost_type_col:
        raise RuntimeError(f"Costs sheet '{COSTS_SHEET}' needs a 'Cost type' column.")

    duration_col = _col_lookup(df, "Duration")  # optional but expected in dummy

    horizon_all = [START_FY + i for i in range(YEARS)]
    year_cols = {int(c): c for c in df.columns if str(c).isdigit()}
    use_cols = [year_cols.get(y, None) for y in horizon_all]

    cut = df[df[cost_type_col].astype(str).map(clean) == str(cost_type).strip()].copy()

    costs_by_project: Dict[str, np.ndarray] = {}
    durations_by_project: Dict[str, List[int]] = {}

    for _, r in cut.iterrows():
        p = clean(r.get(proj_col, ""))
        if not p:
            continue

        seq = np.zeros(len(horizon_all), dtype=float)
        for i, c in enumerate(use_cols):
            if c is None:
                seq[i] = 0.0
            else:
                seq[i] = to_float(r.get(c, 0.0))

        if p in costs_by_project:
            costs_by_project[p] = costs_by_project[p] + seq
        else:
            costs_by_project[p] = seq

        if duration_col:
            d = int(round(to_float(r.get(duration_col, 0.0))))
            if d > 0:
                durations_by_project.setdefault(p, []).append(d)

    costs_input: Dict[str, List[float]] = {}
    for p, seq_dollars in costs_by_project.items():
        costs_input[p] = (seq_dollars / 1_000_000.0).tolist()  # M

    projects: Dict[str, Dict[str, Any]] = {}
    variants: Dict[str, Dict[str, Any]] = {}

    for p, seriesM in costs_input.items():
        s = pd.Series(seriesM, dtype=float)
        nz = np.where(np.abs(s.to_numpy(dtype=float)) > 1e-12)[0]
        if nz.size == 0:
            continue

        first_idx = int(nz.min())

        dur_from_col: Optional[int] = None
        dlist = durations_by_project.get(p, [])
        if dlist:
            # duration = mode (tie-breaker: max)
            counts: Dict[int, int] = {}
            for d in dlist:
                counts[int(d)] = counts.get(int(d), 0) + 1
            dur_from_col = sorted(counts.keys(), key=lambda k: (counts[k], k), reverse=True)[0]

        if dur_from_col is not None and dur_from_col > 0:
            dur = int(dur_from_col)
            seg = s.iloc[first_idx:first_idx + dur].tolist()
            if len(seg) < dur:
                seg = seg + [0.0] * (dur - len(seg))
        else:
            last_idx = int(nz.max())
            seg = s.iloc[first_idx:last_idx + 1].tolist()
            dur = len(seg)

        projects[p] = {"cost": float(sum(seg)), "dur": int(dur), "spend": [float(x) for x in seg]}
        variants[p] = {
            "base": p,
            "dur": int(dur),
            "spend": [float(x) for x in seg],
            "first_year_idx": int(first_idx),
        }

    costs_input_df = pd.DataFrame(costs_input, index=horizon_all).T
    costs_input_df.index.name = "Project"
    return projects, variants, costs_input_df


# ─────────────────────────────────────────────────────────────
# BENEFITS (UPDATED FOR dummy_cost_benefits.xlsx)
# ─────────────────────────────────────────────────────────────
def load_benefits(sheet: str) -> pd.DataFrame:
    """
    Dummy Benefits sheet supported format:
      Project | ... | Dimension | t+0 | t+1 | ... | t+40 (or further)

    - Extra descriptor columns are ignored.
    - t+ columns are robustly parsed to floats.
    """
    df = pd.read_excel(DATA_FILE, sheet_name=sheet, engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    proj_col = _col_lookup(df, "Project")
    if not proj_col:
        raise RuntimeError(f"Benefits sheet '{sheet}' needs 'Project' column.")
    if proj_col != "Project":
        df.rename(columns={proj_col: "Project"}, inplace=True)

    dim_col = None
    for c in df.columns:
        if str(c).strip().lower().startswith("dimension"):
            dim_col = c
            break
    if dim_col is None:
        raise RuntimeError(f"Benefits sheet '{sheet}' needs a 'Dimension' column.")
    if dim_col != "Dimension":
        df.rename(columns={dim_col: "Dimension"}, inplace=True)

    # Identify t+ columns
    tcols: List[Tuple[int, str]] = []
    for c in df.columns:
        m = re.fullmatch(r"[tT]\s*\+\s*(\d+)", str(c))
        if m:
            tcols.append((int(m.group(1)), c))
    tcols.sort(key=lambda x: x[0])

    if not tcols:
        raise RuntimeError(f"Benefits sheet '{sheet}' has no columns matching 't+N' (e.g., t+0..t+40).")

    for _, c in tcols:
        df[c] = df[c].map(to_float).astype(float)

    df["Project_clean"] = df["Project"].map(clean)
    df["Dimension_clean"] = df["Dimension"].map(clean)

    return df


def build_kernels_by_dim(
    benef_df: pd.DataFrame,
    variants: Dict[str, dict],
) -> Tuple[List[str], Dict[str, Dict[str, List[float]]]]:
    """
    Build kernels_by_dim so that:
      - For each Dimension (including 'Total') we have per-project benefit streams.
      - If multiple rows exist for the same (Project, Dimension), their t+ streams are summed.
      - If 'Total' is missing, it's computed as sum of other dims (excluding Total).
      - Each kernel is padded with 'dur' zeros for the construction period.
      - Handles non-contiguous t+ indices (fills missing with zeros).
    """
    # Parse t+ columns with indices
    t_pairs: List[Tuple[int, str]] = []
    for c in benef_df.columns:
        m = re.fullmatch(r"[tT]\s*\+\s*(\d+)", str(c))
        if m:
            t_pairs.append((int(m.group(1)), c))
    t_pairs.sort(key=lambda x: x[0])
    if not t_pairs:
        raise RuntimeError("Benefits DF has no 't+N' columns after loading.")

    tmax = int(t_pairs[-1][0])
    seq_len = tmax + 1

    flows_by_dim: Dict[str, Dict[str, List[float]]] = {}
    dim_order: List[str] = []

    for _, r in benef_df.iterrows():
        p = clean(r.get("Project_clean", ""))
        d = clean(r.get("Dimension_clean", ""))
        if not p or not d:
            continue

        seq = [0.0] * seq_len
        for idx, col in t_pairs:
            seq[idx] = float(to_float(r.get(col, 0.0)))

        flows_by_dim.setdefault(d, {})
        if p in flows_by_dim[d]:
            prev = flows_by_dim[d][p]
            flows_by_dim[d][p] = [a + b for a, b in zip(prev, seq)]
        else:
            flows_by_dim[d][p] = seq

        if d not in dim_order:
            dim_order.append(d)

    # Compute Total if missing
    if "Total" not in flows_by_dim:
        flows_by_dim["Total"] = {}

    other_dims = [d for d in flows_by_dim.keys() if d.lower() != "total"]
    all_projects = set()
    for d in other_dims:
        all_projects |= set(flows_by_dim[d].keys())

    if len(flows_by_dim.get("Total", {})) == 0:
        for p in all_projects:
            acc = [0.0] * seq_len
            for d in other_dims:
                v = flows_by_dim[d].get(p)
                if v is None:
                    continue
                acc = [a + b for a, b in zip(acc, v)]
            flows_by_dim["Total"][p] = acc

    # Final dimension order with Total first (if present)
    dims_unique: List[str] = []
    seen = set()
    for d in dim_order:
        if d not in seen:
            seen.add(d)
            dims_unique.append(d)
    if "Total" not in seen:
        dims_unique.append("Total")
    else:
        dims_unique = ["Total"] + [d for d in dims_unique if d != "Total"]

    # Build padded kernels for only projects that exist in cost variants
    keeps = set(variants.keys())
    kernels_by_dim: Dict[str, Dict[str, List[float]]] = {}

    for d in dims_unique:
        kernels_by_dim[d] = {}
        proj_map = flows_by_dim.get(d, {})
        for v_name, meta in variants.items():
            if v_name not in keeps:
                continue
            dur = int(meta["dur"])
            seq = proj_map.get(v_name, [0.0] * seq_len)
            kernels_by_dim[d][v_name] = [0.0] * dur + [float(x) for x in seq]

    return dims_unique, kernels_by_dim


# ─────────────────────────────────────────────────────────────
# SELECTION FROM modelTOT.csv (EXACT SAME CSV)
# ─────────────────────────────────────────────────────────────
def _find_column(df: pd.DataFrame, *candidates: str) -> Optional[str]:
    """
    Try to find a column in df whose name matches any of the candidate names,
    ignoring spaces/underscores and case. If no exact match, falls back to
    'contains' logic.
    """
    norm_map = {re.sub(r"[\s_]+", "", str(c)).lower(): c for c in df.columns}

    for cand in candidates:
        key = re.sub(r"[\s_]+", "", cand).lower()
        if key in norm_map:
            return str(norm_map[key])

    for cand in candidates:
        ck = re.sub(r"[\s_]+", "", cand).lower()
        for c in df.columns:
            c_norm = re.sub(r"[\s_]+", "", str(c)).lower()
            if ck in c_norm:
                return str(c)

    return None


def _maybe_filter(df: pd.DataFrame, col: Optional[str], target: Any, *, kind: str) -> pd.DataFrame:
    """
    Filter df to rows where df[col] matches target.
    If col is None, returns df unchanged.
    """
    if col is None or col not in df.columns:
        return df

    if kind == "text":
        tgt = clean(str(target)).lower()
        return df[df[col].astype(str).map(lambda x: clean(x).lower()) == tgt].copy()

    if kind == "buffer":
        tgt_int = int(target)

        def match(v: Any) -> bool:
            if v is None:
                return False
            if isinstance(v, (int, float, np.integer, np.floating)):
                try:
                    return int(round(float(v))) == tgt_int
                except Exception:
                    return False
            s = clean(str(v)).lower()
            if s == str(tgt_int):
                return True
            # match things like "pm0", "PM0", "+0", "0.0"
            s2 = re.sub(r"\s+", "", s)
            if s2 in {f"pm{tgt_int}", f"+{tgt_int}", f"{tgt_int}.0"}:
                return True
            # extract last integer if present
            m = re.search(r"-?\d+", s2)
            if m:
                try:
                    return int(m.group(0)) == tgt_int
                except Exception:
                    return False
            return False

        return df[df[col].map(match)].copy()

    return df


def load_selection_from_modelTOT(variants: Dict[str, dict]) -> Dict[Tuple[str, int], int]:
    """
    Reads modelTOT.csv and converts it into:
        {(ProjectName, start_index): 1}
    where start_index = StartYear - START_FY.

    Robust enhancements:
      - Detects Project/StartYear/Duration columns with flexible names
      - If run/buffer/dimension/cost-type columns exist, filters using MODEL_* config.
      - Normalises project names to match cost variants via norm_key mapping.
    """
    if not MODEL_TOT_FILE.exists():
        raise FileNotFoundError(f"Model CSV not found: {MODEL_TOT_FILE}")

    df = pd.read_csv(MODEL_TOT_FILE)
    df.columns = [str(c).strip() for c in df.columns]

    # Optional filters (applied only if columns exist)
    run_col = _find_column(df, "Run", "ModelRun", "MODEL_RUN", "SurplusKey", "FundingKey", "EnvelopeKey")
    buf_col = _find_column(df, "Buffer", "PlusMinus", "pm", "PLUS_MINUS", "Delta", "BufferM")
    dim_col = _find_column(df, "Dimension", "PrimaryDim", "Objective", "Dim")
    cost_col = _find_column(df, "CostScenario", "CostType", "Cost type", "COST_TYPE", "Cost")

    df0 = df
    df0 = _maybe_filter(df0, run_col, MODEL_RUN, kind="text")
    df0 = _maybe_filter(df0, buf_col, MODEL_BUFFER, kind="buffer")
    df0 = _maybe_filter(df0, dim_col, MODEL_DIMENSION, kind="text")
    df0 = _maybe_filter(df0, cost_col, MODEL_COST_SCENARIO, kind="text")

    # Required columns for schedule
    proj_col = _find_column(df0, "Project", "ProjectName", "Project_Name")
    start_col = _find_column(df0, "StartYear", "StartFY", "Start_FY", "Start")
    dur_col = _find_column(df0, "Duration", "Dur", "ProjectDuration")  # optional

    if not proj_col or not start_col:
        raise RuntimeError(
            "modelTOT.csv must contain at least Project and StartYear/StartFY columns "
            f"(found Project={proj_col!r}, Start={start_col!r})."
        )

    # Map normalised project name → variant name from the cost data
    norm_to_variant: Dict[str, str] = {norm_key(v): v for v in variants.keys()}

    sel: Dict[Tuple[str, int], int] = {}
    seen_projects: Dict[str, int] = {}

    for _, r in df0.iterrows():
        proj_raw = str(r.get(proj_col, ""))
        if not clean(proj_raw):
            continue

        try:
            start_year = int(round(to_float(r.get(start_col, 0))))
        except Exception:
            continue

        key = norm_key(proj_raw)
        proj_canon = norm_to_variant.get(key)
        if proj_canon is None:
            print(f"Warning: CSV project {proj_raw!r} not found in cost data; skipping.")
            continue

        idx = start_year - START_FY
        if idx < 0 or idx >= TFIXED:
            print(
                f"Warning: project {proj_canon!r} has StartYear {start_year}, "
                f"outside horizon {START_FY}-{START_FY + TFIXED - 1}; skipping."
            )
            continue

        # Optional duration check
        if dur_col and dur_col in df0.columns:
            csv_dur = int(round(to_float(r.get(dur_col, 0))))
            model_dur = int(variants[proj_canon]["dur"])
            if csv_dur > 0 and csv_dur != model_dur:
                print(
                    f"Warning: duration mismatch for {proj_canon!r}: "
                    f"CSV={csv_dur} vs Costs-derived={model_dur}. Using Costs-derived."
                )

        # Warn if duplicate project appears with different start
        if proj_canon in seen_projects and seen_projects[proj_canon] != idx:
            print(
                f"Warning: project {proj_canon!r} appears multiple times in CSV with different starts "
                f"({START_FY + seen_projects[proj_canon]} and {start_year}). Using the latest row."
            )

        seen_projects[proj_canon] = idx
        sel[(proj_canon, idx)] = 1

    return sel


# ─────────────────────────────────────────────────────────────
# MAIN PKL BUILDER
# ─────────────────────────────────────────────────────────────
def build_and_save_pickle_from_sol() -> Path:
    print(f"Reading costs & benefits from: {DATA_FILE}")
    print(f"  Costs sheet: {COSTS_SHEET}")
    print(f"  Benefits sheet: {BENEFITS_SHEET}")

    # 1) Costs + benefits from dummy workbook
    projects_all, variants_all, costs_input_df = load_costs(MODEL_COST_SCENARIO)
    benef_df = load_benefits(BENEFITS_SHEET)
    dims_order, kernels_by_dim = build_kernels_by_dim(benef_df, variants_all)

    # Keep all projects with costs (same style as optimiser outputs)
    projects = {p: projects_all[p] for p in variants_all.keys() if p in projects_all}

    # 2) Selection from the SAME modelTOT.csv schedule
    print(f"Reading schedule from CSV: {MODEL_TOT_FILE}")
    sel_raw = load_selection_from_modelTOT(variants_all)
    if not sel_raw:
        raise RuntimeError("No selected projects found in modelTOT.csv after filtering/matching.")

    # Filter to projects with cost data
    sel: Dict[Tuple[str, int], int] = {}
    for (p, s), on in sel_raw.items():
        if on and p in variants_all:
            sel[(p, s)] = 1
        elif on and p not in variants_all:
            print(f"Warning: mapped project {p!r} has no cost entry; skipping.")

    if not sel:
        raise RuntimeError("After filtering, no modelTOT projects matched cost data.")

    ny = TFIXED
    fy = cal_years(ny)
    funding_target_M = np.full(ny, float(FULL_ENVELOPE_M), dtype=float)

    print(
        f"Horizon: {ny} years ({START_FY}–{START_FY + ny - 1}), "
        f"envelope {FULL_ENVELOPE_M:.0f} M/yr"
    )
    print(f"Selected projects: {len(sel)}")
    print(f"Dimensions found (incl Total if present/computed): {dims_order}")

    # 3) Schedule table
    sched_rows = []
    for (v, s) in sorted(sel.keys(), key=lambda x: (x[1], x[0])):
        dur = int(variants_all[v]["dur"])
        sched_rows.append(
            {
                "Project": v,
                "StartFY": START_FY + s,
                "EndFY": START_FY + s + dur - 1,
                "Dur": dur,
                "Scenario": "LIN40",     # keep label if downstream expects it
                "PrimaryDim": "Total",
            }
        )
    df_sched = pd.DataFrame(sched_rows).sort_values(["StartFY", "Project"], ignore_index=True)

    # 4) Spend per project/year (M)
    df_sp = pd.DataFrame(0.0, index=list(projects.keys()), columns=fy)
    for (v, s) in sel.keys():
        vec = variants_all[v]["spend"]
        for i, amt in enumerate(vec):
            t = s + i
            if 0 <= t < ny:
                df_sp.loc[v, fy[t]] += float(amt)
    df_sp.loc["Total Spend"] = df_sp.sum()

    # Last year with spend
    total_spend_series = df_sp.loc["Total Spend"]
    last_spend_year = None
    for yr in reversed(fy):
        if abs(float(total_spend_series.get(yr, 0.0))) > 1e-6:
            last_spend_year = yr
            break
    last_spend_idx = -1 if last_spend_year is None else fy.index(last_spend_year)

    # 5) Funding / net (no dividends)
    funding_M_raw = funding_target_M.tolist()
    dividend_M_raw = [0.0] * ny

    net_M_raw: List[float] = []
    net_prev = 0.0
    for t in range(ny):
        yr = fy[t]
        spend = float(df_sp.loc["Total Spend", yr])
        net_now = net_prev + funding_M_raw[t] - spend - dividend_M_raw[t]
        net_M_raw.append(net_now)
        net_prev = net_now

    funding_M: List[float] = []
    net_M: List[float] = []
    env_M_plot: List[float] = []
    div_M: List[float] = []

    for t in range(ny):
        if last_spend_idx >= 0 and t <= last_spend_idx:
            fund = funding_M_raw[t]
            netv = net_M_raw[t]
            env = float(funding_target_M[t])
            div = dividend_M_raw[t]
        else:
            fund = 0.0
            netv = 0.0
            env = 0.0
            div = 0.0
        funding_M.append(fund)
        net_M.append(netv)
        env_M_plot.append(env)
        div_M.append(div)

    cash_rows = []
    for t in range(ny):
        yr = fy[t]
        spend = float(df_sp.loc["Total Spend", yr])
        env = env_M_plot[t]
        fund = funding_M[t]
        opening_net = 0.0 if t == 0 else net_M[t - 1]
        closing_net = net_M[t]
        div = div_M[t]
        cash_rows.append(
            dict(
                Year=yr,
                Envelope=env,
                Funding=fund,
                OpeningNet=opening_net,
                Spend=spend,
                Dividend=div,
                ClosingNet=closing_net,
                OpeningCash=max(opening_net, 0.0),
                OpeningDebt=max(-opening_net, 0.0),
                ClosingCash=max(closing_net, 0.0),
                ClosingDebt=max(-closing_net, 0.0),
            )
        )
    df_cash = pd.DataFrame(cash_rows)

    # 6) Benefits streams & PVs (40-year PV window)
    dims = list(kernels_by_dim.keys())

    Tstore = ny
    for (v, s) in sel.keys():
        for d in dims:
            ker = kernels_by_dim.get(d, {}).get(v, [])
            if ker:
                Tstore = max(Tstore, s + len(ker))

    fy_store = cal_years(Tstore)

    ben_total_by_dim_store: Dict[str, np.ndarray] = {d: np.zeros(Tstore, float) for d in dims}
    proj_dim_year_store: Dict[Tuple[str, str], np.ndarray] = {
        (p, d): np.zeros(Tstore, float) for d in dims for p in projects.keys()
    }

    for (v, s) in sel.keys():
        for d in dims:
            ker = kernels_by_dim[d].get(v, [])
            for k, f in enumerate(ker):
                t = s + k
                if 0 <= t < Tstore:
                    ben_total_by_dim_store[d][t] += float(f)
                    proj_dim_year_store[(v, d)][t] += float(f)

    # Raw benefit streams
    df_ben_year = pd.DataFrame({"Year": fy_store, **{d: ben_total_by_dim_store[d] for d in dims}})

    idx = pd.MultiIndex.from_product([list(projects.keys()), dims], names=["Project", "Dimension"])
    df_ben_proj_dim_year = pd.DataFrame(0.0, index=idx, columns=fy_store)
    for (p, d), vec in proj_dim_year_store.items():
        df_ben_proj_dim_year.loc[(p, d), :] = vec

    # Discounting only for PV (40-year window)
    disc = np.array([(1.0 + BENEFIT_DISCOUNT_RATE) ** t for t in range(ny)], float)
    pv_by_dim: Dict[str, float] = {d: float(np.sum(ben_total_by_dim_store[d][:ny] / disc)) for d in dims}

    pv_by_proj_dim = pd.DataFrame(0.0, index=idx, columns=["PV"])
    for (p, d) in idx:
        vec_full = df_ben_proj_dim_year.loc[(p, d)].to_numpy(dtype=float)
        pv_by_proj_dim.loc[(p, d), "PV"] = float(np.sum(vec_full[:ny] / disc))

    total_pv = pv_by_dim["Total"] if "Total" in pv_by_dim else float(
        np.sum([pv for d, pv in pv_by_dim.items() if d.lower() != "total"])
    )

    # Diagnostics
    closing_net_M_raw = float(net_M_raw[-1]) if net_M_raw else 0.0
    max_net_M_raw = float(max(net_M_raw)) if net_M_raw else 0.0
    diag_caps: Dict[str, Any] = {
        "closing_net_M_raw": closing_net_M_raw,
        "max_net_M_raw": max_net_M_raw,
        "source_csv": str(MODEL_TOT_FILE),
        "source_xlsx": str(DATA_FILE),
        "costs_sheet": COSTS_SHEET,
        "benefits_sheet": BENEFITS_SHEET,
        "pv_horizon_years": ny,
    }

    # Build PKL payload
    gap_now = None
    gap_pct = None

    out: Dict[str, Any] = dict(
        status="OK",
        objective="Total",
        created_at=_now_stamp(),
        scenario="LIN40",         # keep label if downstream expects it
        primary_dim="Total",
        schedule=df_sched,
        spend=df_sp,
        cash_flow=df_cash,
        envelope=pd.DataFrame({"Year": fy, "Envelope": env_M_plot}),
        benefits_by_year=df_ben_year,
        benefits_by_project_dimension_by_year=df_ben_proj_dim_year,
        pv_by_dimension=pv_by_dim,
        pv_total=total_pv,
        gap=gap_now,
        gap_pct=gap_pct,
        best={"pv": total_pv, "gap": gap_now, "gap_pct": gap_pct},
        pv_by_project_and_dimension=pv_by_proj_dim,
        calendar=dict(start_fy=START_FY, years=ny),
        meta=dict(
            full_envelope_M=float(FULL_ENVELOPE_M),
            taper_years=0,
            backlog_weight=BACKLOG_WEIGHT,
            pv_weight=PV_WEIGHT,
            net_cap_tiers=PIECEWISE_CAP_TIERS,
            run_id=RUN_ID,
            model_run=MODEL_RUN,
            model_buffer=MODEL_BUFFER,
            model_dimension=MODEL_DIMENSION,
            model_cost_scenario=MODEL_COST_SCENARIO,
        ),
        diagnostic=diag_caps,
    )

    out_path = CACHE / PKL_NAME
    pkl_save(out_path, out)
    print("Total PV (Total dimension):", f"{total_pv:,.6f}")
    return out_path


if __name__ == "__main__":
    build_and_save_pickle_from_sol()


Reading costs & benefits from: C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation\dummy_cost_benefits.xlsx
  Costs sheet: Costs
  Benefits sheet: Benefits
Reading schedule from CSV: C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation\modelTOT.csv
Horizon: 40 years (2026–2065), envelope 1000 M/yr
Selected projects: 50
Dimensions found (incl Total if present/computed): ['Total', 'Economic Prosperity', 'Healthy and safe people', 'Inclusive Access']
Saved: C:\Users\Adrian Desilvestro\Documents\NZTA\Project_Rons_optimisation\scenario_cache_benefit_mo\HK_dummy_P50Real_DUMMY_s1000_pm0_TOT.pkl (117600 bytes) objective=Total pv=30729723332.316315 gap=n/a
Total PV (Total dimension): 30,729,723,332.316315
